In [80]:
# ============================================================
# Imports
# ============================================================

# Standard library
import os
import re
import copy
import time
import random
import shutil
from pathlib import Path
from collections import Counter, defaultdict

# Data handling
import numpy as np
import pandas as pd
import joblib

# Image handling / visualization
from PIL import Image
import pillow_heif
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn metrics
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
)

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models
import torchvision.transforms.v2 as v2
from torch.utils.data import (
    DataLoader,
    Subset,
    ConcatDataset,
)

# Environment configuration
from dotenv import load_dotenv


# Enable HEIF/HEIC/AVIF image support
pillow_heif.register_heif_opener()

print("=== Environment Info ===")
print(f"OS: {os.name}")
print(f"Python version: {os.sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")

if torch.backends.mps.is_available():
    print("Apple GPU (MPS) available: Yes")
else:
    print("Apple GPU (MPS) available: No")

=== Environment Info ===
OS: posix
Python version: 3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:54:21) [Clang 16.0.6 ]
PyTorch version: 2.5.1
Torchvision version: 0.20.1
Numpy version: 1.26.4
Pandas version: 2.2.2
Matplotlib version: 3.9.2
Seaborn version: 0.13.2
Apple GPU (MPS) available: Yes


In [4]:
# ============================================================
# Project / Dataset Paths
# ============================================================

from dotenv import load_dotenv

# Permanent private configuration
CONFIG_FILE = Path.home() / ".wadidegla.env"

if not CONFIG_FILE.exists():
    raise FileNotFoundError(
        "Local WadiDegla configuration file was not found."
    )

load_dotenv(CONFIG_FILE)

# Wadi Degla dataset
DATA_DIR = Path(os.environ["WADIDEGLA_DATA_DIR"])
TRAIN_DIR = DATA_DIR / "Train"
VAL_DIR = DATA_DIR / "Validation"

# External web dataset
WEB_DATA_DIR = Path(os.environ["WEB_DATA_DIR"])

# Experimental workspace
WORKSPACE_DIR = Path(os.environ["WADIDEGLA_WORKSPACE_DIR"])


if not DATA_DIR.exists():
    raise FileNotFoundError(
        "The configured WadiDegla dataset directory does not exist."
    )

if not WEB_DATA_DIR.exists():
    raise FileNotFoundError(
        "The configured external web dataset directory does not exist."
    )

if not WORKSPACE_DIR.exists():
    raise FileNotFoundError(
        "The configured WadiDegla experimental workspace directory does not exist."
    )

print("WadiDegla dataset configuration loaded successfully.")
print("External web dataset configuration loaded successfully.")
print("WadiDegla experimental workspace configuration loaded successfully.")

WadiDegla dataset configuration loaded successfully.
External web dataset configuration loaded successfully.
WadiDegla experimental workspace configuration loaded successfully.


# <center>**Utility Functions for Model Training & Evaluation**</center>

## **1. Compute Class Weights**

In [5]:
def compute_class_weights(dataset, method="inverse_frequency", normalize=False):
    """
    Compute class weights based on the chosen method, with optional normalization.

    Args:
        dataset: A dataset object with a 'targets' attribute containing class labels.
        method (str): Method for computing class weights. Options:
            - "inverse_frequency": total_samples / count
            - "balanced": total_samples / (count * num_classes)
            - "sqrt_inverse": 1 / np.sqrt(count)
            - "scaled_sqrt_inverse": (1 / np.sqrt(count)) * 100
        normalize (bool): If True, rescale weights so that their mean = 1.

    Returns:
        torch.Tensor: Class weights tensor on the appropriate device.
    """
    # Compute class frequencies
    class_counts = Counter(dataset.targets)
    total_samples = sum(class_counts.values())
    num_classes = len(class_counts)

    # Select weight calculation method
    if method == "inverse_frequency":
        class_weights = {cls: total_samples / count for cls, count in class_counts.items()}
    elif method == "balanced":
        class_weights = {cls: total_samples / (count * num_classes) for cls, count in class_counts.items()}
    elif method == "sqrt_inverse":
        class_weights = {cls: 1 / np.sqrt(count) for cls, count in class_counts.items()}
    elif method == "scaled_sqrt_inverse":
        class_weights = {cls: (1 / np.sqrt(count)) * 100 for cls, count in class_counts.items()}
    else:
        raise ValueError("Invalid method. Choose from 'inverse_frequency', 'balanced', 'sqrt_inverse', 'scaled_sqrt_inverse'.")

    # Convert to tensor
    class_weights_tensor = torch.tensor([class_weights[i] for i in range(num_classes)], dtype=torch.float32)

    #  Normalize weights to have an average of 1.0 → divide by mean 
    if normalize:
        class_weights_tensor = class_weights_tensor / class_weights_tensor.mean()

    # Move weights to MPS (Apple GPU) if available
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    return class_weights_tensor.to(device)

## **2. Candidate Models Configuration**

### **1. mobilenet_v2**

In [8]:
def summarize_mobilenetv2_blocks(model: nn.Module):
    """
    Summarize blocks in a MobileNetV2 model.
    
    Returns:
        pd.DataFrame: block-level summary with counts, percentages, and cumulative stats.
    """

    # ---- Collect blocks: features + classifier ----
    blocks = []
    if hasattr(model, "features"):
        for i, block in enumerate(model.features):
            blocks.append((f"{block.__class__.__name__}_{i+1}", block, "features"))
    if hasattr(model, "classifier"):
        blocks.append((f"{model.classifier.__class__.__name__}", model.classifier, "classifier"))

    # ---- Helper functions ----
    def count_layers(module: nn.Module):
        # Count atomic submodules (ignore containers)
        return sum(1 for m in module.modules() if not isinstance(m, nn.Sequential))

    def count_params(module: nn.Module):
        return sum(p.numel() for p in module.parameters())

    def count_trainable_params(module: nn.Module):
        return sum(p.numel() for p in module.parameters() if p.requires_grad)

    # ---- Build dataframe rows ----
    rows = []
    for order, (name, block, module_name) in enumerate(blocks, 1):
        n_layers = count_layers(block)
        n_params = count_params(block)
        n_trainable = count_trainable_params(block)
        rows.append({
            "Block": name,
            "Order": order,
            "Module": module_name,
            "Layers (#)": n_layers,
            "Params (#)": n_params,
            "Trainable Params (#)": n_trainable,
        })

    df = pd.DataFrame(rows)

    # ---- Totals for whole model ----
    total_layers_model = df["Layers (#)"].sum()
    total_params_model = df["Params (#)"].sum()
    total_trainable_params_model = df["Trainable Params (#)"].sum()

    # Per-model percentages
    df["% Layers (Model)"] = df["Layers (#)"] / total_layers_model * 100
    df["% Params (Model)"] = df["Params (#)"] / total_params_model * 100
    df["% Trainable Params (Model)"] = df["Trainable Params (#)"] / total_params_model * 100

    # Accumulative (start from deepest → reverse order)
    df["Cumulative Layers (Model)"] = df["Layers (#)"][::-1].cumsum()[::-1]
    df["Cumulative % Layers (Model)"] = df["Cumulative Layers (Model)"] / total_layers_model * 100
    df["Cumulative Params (Model)"] = df["Params (#)"][::-1].cumsum()[::-1]
    df["Cumulative % Params (Model)"] = df["Cumulative Params (Model)"] / total_params_model * 100
    # ---- Per-module stats ----
    module_stats = []
    for module_name in df["Module"].unique():
        sub = df[df["Module"] == module_name]
        total_layers_mod = sub["Layers (#)"].sum()
        total_params_mod = sub["Params (#)"].sum()
        total_trainable_mod = sub["Trainable Params (#)"].sum()
        for idx in sub.index:
            module_stats.append({
                "Layers (Module %)": df.loc[idx, "Layers (#)"] / total_layers_mod * 100,
                "Params (Module %)": df.loc[idx, "Params (#)"] / total_params_mod * 100,
                "Trainable Params (Module %)": df.loc[idx, "Trainable Params (#)"] / total_trainable_mod * 100 if total_trainable_mod > 0 else 0,
                "Cumulative Layers (Module)": sub["Layers (#)"][::-1].cumsum()[::-1].loc[idx],
                "Cumulative % Layers (Module)": sub["Layers (#)"][::-1].cumsum()[::-1].loc[idx] / total_layers_mod * 100,
                "Cumulative Params (Module)": sub["Params (#)"][::-1].cumsum()[::-1].loc[idx],
                "Cumulative % Params (Module)": sub["Params (#)"][::-1].cumsum()[::-1].loc[idx] / total_params_mod * 100,
            })
    module_df = pd.DataFrame(module_stats, index=df.index)

    # Combine
    df = pd.concat([df, module_df], axis=1)

    return df

## 3. Model Training

In [9]:
def train_model(model, train_loader, criterion, optimizer, class_names, epoch_num=1, log_interval=100):
    start_time = time.time()  # Start tracking time
    device = torch.device("cuda" if torch.cuda.is_available() else "mps")  # Use GPU if available on Mac

    # Dictionary to store per-class metrics
    class_metrics = defaultdict(lambda: {"correct": 0, "total": 0, "y_true": [], "y_pred": []})

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    epoch_start_time = time.time()  # Track epoch time

    for batch_idx, (images, labels) in enumerate(train_loader, 1):  # Start index from 1
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Compute accuracy
        _, predicted = outputs.max(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        accuracy = 100 * correct / total

        # Store predictions for per-class metrics
        for label, pred in zip(labels.cpu().numpy(), predicted.cpu().numpy()):
            class_metrics[label]["correct"] += (label == pred)
            class_metrics[label]["total"] += 1
            class_metrics[label]["y_true"].append(label)
            class_metrics[label]["y_pred"].append(pred)

        # Print loss and accuracy every 100 batches
        if batch_idx % log_interval == 0:
            elapsed_time = (time.time() - start_time)/60
            print(f"Epoch {epoch_num}, Batch {batch_idx}, Loss: {running_loss/batch_idx:.4f}, "
                  f"Accuracy: {accuracy:.2f}%, Time Passed: {elapsed_time:.2f}m")

    epoch_time = (time.time() - epoch_start_time)/60
    print(f"Epoch {epoch_num} Completed - Average Loss: {running_loss/len(train_loader):.4f}, "
          f"Accuracy: {accuracy:.2f}%, Epoch Time: {epoch_time:.2f}m")
    overall_accuracy = 100 * correct / total

    total_time = (time.time() - start_time)/60
    print(f"Training Complete Epoch {epoch_num} - Total Time: {total_time:.2f}m")

    # Calculate per-class accuracy, precision, recall, and F1-score
    all_y_true = []
    all_y_pred = []
    
    for class_idx in sorted(class_metrics.keys()):
        all_y_true.extend(class_metrics[class_idx]["y_true"])
        all_y_pred.extend(class_metrics[class_idx]["y_pred"])
    
    # Compute per-class precision, recall, and F1-score
    unique_classes = sorted(set(all_y_true))  # Get all unique classes
    per_class_metrics = {
        "precision": precision_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0),
        "recall": recall_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0),
        "f1_score": f1_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0)
    }

    # Store per-class results in alignment with unique_classes
    class_results = []
    for class_idx in unique_classes:
        class_results.append([
            class_names[class_idx],
            per_class_metrics["precision"][unique_classes.index(class_idx)]*100,
            per_class_metrics["recall"][unique_classes.index(class_idx)]*100,
            per_class_metrics["f1_score"][unique_classes.index(class_idx)]*100
        ])

    # Create a DataFrame with per-class accuracy, precision, recall, and F1-score
    df_results = pd.DataFrame(class_results, columns=["Species", "Precision", "Recall", "F1-score"])

    # Append overall metrics (Macro) as the last row
    macro_precision = per_class_metrics["precision"].mean()*100
    macro_recall = per_class_metrics["recall"].mean()*100
    macro_f1 = per_class_metrics["f1_score"].mean()*100

    df_results.loc[len(df_results)] = ["Average Performance (Macro)", macro_precision, macro_recall, macro_f1]

    df_results['Epoch'] = epoch_num
    avg_loss = running_loss / len(train_loader)

    return {
    'model': model,
    'optimizer': optimizer,
    'metrics_df': df_results,
    'overall_accuracy': overall_accuracy,
    'total_loss': running_loss,
    'avg_loss': avg_loss,
    'epoch_time': epoch_time
            }

## 4. Model Evaluation

In [10]:
def evaluate_model(model, val_loader, criterion, class_names, epoch_num=1, log_interval=100):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "mps")
    
    start_time = time.time()  # Track evaluation time

    # Dictionary to store per-class metrics
    class_metrics = defaultdict(lambda: {"correct": 0, "total": 0, "y_true": [], "y_pred": []})
    all_y_true = []
    all_y_pred = []
    correct = 0
    total = 0
    running_loss = 0.0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(val_loader, 1):  # Start index from 1
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            # Compute loss
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            
            _, preds = torch.max(outputs, 1)

            all_y_true.extend(labels.cpu().numpy())
            all_y_pred.extend(preds.cpu().numpy())

            # Compute accuracy
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            accuracy = 100 * correct / total

            # Store per-class metrics
            for label, pred in zip(labels.cpu().numpy(), preds.cpu().numpy()):
                class_metrics[label]["correct"] += (label == pred)
                class_metrics[label]["total"] += 1
                class_metrics[label]["y_true"].append(label)
                class_metrics[label]["y_pred"].append(pred)

            # Print progress every 100 batches
            if batch_idx % log_interval == 0:
                elapsed_time = (time.time() - start_time)/60
                avg_loss = running_loss / batch_idx
                print(f"Batch {batch_idx}, Loss: {avg_loss:.4f}, "
                      f"Accuracy: {accuracy:.2f}%, Time Passed: {elapsed_time:.2f}m")

    total_time = (time.time() - start_time)/60
    avg_loss = running_loss / len(val_loader)
    print(f"Evaluation Complete - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%, Total Time: {total_time:.2f}m")
    overall_accuracy = 100 * correct / total

    # Compute per-class accuracy
    class_results = []
    for class_idx in sorted(class_metrics.keys()):
        metrics = class_metrics[class_idx]
        class_results.append([class_names[class_idx]])

    # Compute per-class precision, recall, and F1-score
    unique_classes = sorted(set(all_y_true))  # Get all unique classes
    per_class_metrics = {
        "precision": precision_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0),
        "recall": recall_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0),
        "f1_score": f1_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0)
    }

    # Store per-class metrics
    for idx, class_label in enumerate(unique_classes):
        class_results[idx].extend([
            per_class_metrics["precision"][idx],
            per_class_metrics["recall"][idx],
            per_class_metrics["f1_score"][idx]
        ])

    # Create a DataFrame with per-class accuracy, precision, recall, and F1-score
    df_results = pd.DataFrame(class_results, columns=["Species", "Precision", "Recall", "F1-score"])

    # Append overall metrics (Macro) as the last row
    macro_precision = per_class_metrics["precision"].mean()
    macro_recall = per_class_metrics["recall"].mean()
    macro_f1 = per_class_metrics["f1_score"].mean()

    df_results.loc[len(df_results)] = ["Average Performance (Macro)", macro_precision, macro_recall, macro_f1]

    df_results['Epoch'] = epoch_num
    return {
    'metrics_df': df_results,
    'overall_accuracy': overall_accuracy,
    'avg_loss': avg_loss,
    'epoch_time': total_time
            }

## 5. Create performance summary

> **Legacy terminology note:** The column name `Train & Test` is retained from the historical implementation. In Phase 2, the relevant WadiDegla subset is the **validation set**, not an independent test set.

In [11]:
def create_performance_summary(df, micro_accuracy, avg_loss, model_name, train_val, class_weights, time_taken, epoch):
    # Extract macro metrics from the last row
    
    macro_row = df[df['Species'] == 'Average Performance (Macro)']
    precision_macro = macro_row['Precision'].values[0]
    recall_macro = macro_row['Recall'].values[0]
    f1_macro = macro_row['F1-score'].values[0]


    # Exclude the last row to analyze per-class accuracies
    class_accuracies = df[df['Species'] != 'Average Performance (Macro)'].copy()

    # Identify best and worst class based on accuracy
    best_class = class_accuracies.loc[class_accuracies['F1-score'].idxmax(), 'Species']
    worst_class = class_accuracies.loc[class_accuracies['F1-score'].idxmin(), 'Species']

    # Compute the median accuracy and find the nearest class
    median_f1_macro = class_accuracies['F1-score'].median()
    class_accuracies['Abs_Diff'] = (class_accuracies['F1-score'] - median_f1_macro).abs()
    median_class = class_accuracies.loc[class_accuracies['Abs_Diff'].idxmin(), 'Species']

    # Compute the number of classes above micro and macro accuracy averages
    num_classes_above_f1_macro_avg = (class_accuracies['F1-score'] > f1_macro).sum()
    num_classes_above_f1_macro_median = (class_accuracies['F1-score'] > median_f1_macro).sum()

    # Create the final summary DataFrame
    performance_summary = pd.DataFrame({
        "Model": [model_name],
        "Train & Test": [train_val],
        "Accuracy (micro)": [micro_accuracy],
        "Precision (macro)": [precision_macro],
        "Recall (macro)": [recall_macro],
        "F1 (macro)": [f1_macro],
        "Loss": [avg_loss],
        "Class_weights": [class_weights],
        "Best_class": [best_class],
        "Median_class": [median_class],
        "Worst_class": [worst_class],
        "num_classes_above_f1_macro_avg": [num_classes_above_f1_macro_avg],
        "num_classes_above_f1_macro_median": [num_classes_above_f1_macro_median],
        "Time Taken (mins)": [time_taken],
        "Epoch": [epoch]
    })


    return performance_summary

## 6. Model Training & Evaluation Loop

In [12]:
def train_and_evaluate_model(
    model: nn.Module,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    class_names,
    start_epoch: int,
    end_epoch: int,
    model_name: str,
    save_dir: str,
    class_weights: str,
    log_interval: int,
    evaluate = True
):
    """
    Train and evaluate a model, storing per-epoch metrics, summaries, and states.

    Returns:
        dict: {
            'epoch_states': {epoch_num: {'model_state': ..., 'optimizer_state': ..., 'train_metrics': ..., 'val_metrics': ...}},
            'train_results_df': pd.DataFrame,
            'train_summary_df': pd.DataFrame,
            'val_results_df': pd.DataFrame,
            'val_summary_df': pd.DataFrame
        }
    """
    device = torch.device("mps")
    model.to(device)  # Move model to MPS

    # Initialize storage
    epoch_states = {}
    train_results_list, train_summary_list = [], []
    val_results_list, val_summary_list = [], []


    for epoch in range(start_epoch, end_epoch + 1):
        print(f"\n🚀 Epoch {epoch} - Training Started...\n")
        
        # ---- Training ----
        training_result = train_model(
            model, train_loader, criterion, optimizer, class_names, epoch_num=epoch, log_interval=log_interval
        )
        train_summary = create_performance_summary(
            training_result['metrics_df'],
            training_result['overall_accuracy'],
            training_result['avg_loss'],
            model_name=model_name,
            train_val='Train',
            class_weights=class_weights,
            time_taken=training_result['epoch_time'],
            epoch = epoch
        )
        
        train_results_list.append(training_result['metrics_df'])
        train_summary_list.append(train_summary)
        if evaluate:

            print(f"\n✅ Epoch {epoch} - Training Completed. Starting Evaluation...\n")
    
            # ---- Validation ----
            val_result = evaluate_model(
                model, val_loader, criterion, class_names, epoch_num=epoch, log_interval=log_interval
            )
            val_summary = create_performance_summary(
                val_result['metrics_df'],
                val_result['overall_accuracy'],
                val_result['avg_loss'],
                model_name=model_name,
                train_val='Validation',
                class_weights=class_weights,
                time_taken=val_result['epoch_time'],
                epoch = epoch
            )
    
            val_results_list.append(val_result['metrics_df'])
            val_summary_list.append(val_summary)
            print(f"\n📊 Epoch {epoch} - Evaluation Completed.\n")

        else:
            print(f"\n✅ Epoch {epoch} - Training Completed (No Evaluation).\n")

        # ---- Save model + optimizer state for this epoch ----
        epoch_states[epoch] = {
            'model_state_dict': copy.deepcopy(model.state_dict()),
            'optimizer_state_dict': copy.deepcopy(optimizer.state_dict())
        }

        

    # ---- Combine DataFrames ----
    train_results_df = pd.concat(train_results_list, ignore_index=True)
    train_summary_df = pd.concat(train_summary_list, ignore_index=True)
    if evaluate:
        val_results_df = pd.concat(val_results_list, ignore_index=True)
        val_summary_df = pd.concat(val_summary_list, ignore_index=True)

    else:
        val_results_df, val_summary_df = None, None

    # Optionally save to CSV (you can remove if not needed)
    train_results_df.to_csv(f"{save_dir}/{model_name}_train_results{start_epoch}_{end_epoch}.csv", index=False)
    train_summary_df.to_csv(f"{save_dir}/{model_name}_train_summary{start_epoch}_{end_epoch}.csv", index=False)
    if evaluate:
        val_results_df.to_csv(f"{save_dir}/{model_name}_val_results{start_epoch}_{end_epoch}.csv", index=False)
        val_summary_df.to_csv(f"{save_dir}/{model_name}_val_summary{start_epoch}_{end_epoch}.csv", index=False)

    return epoch_states, {
        'train_results_df': train_results_df,
        'train_summary_df': train_summary_df,
        'val_results_df': val_results_df,
        'val_summary_df': val_summary_df
    }

## **7. Testing Model Generalization on Web Images from data dir or data loader**

> **Terminology note:** In this notebook, references to "testing" on web images denote external web evaluation during model development. The web data do not constitute an independent test set.

In [13]:
def evaluate_models_on_directory(models_dict, data_dir, class_names,
                                 display_images=False, particular_model=None, Minimum=None):
    """
    Evaluate multiple models on a directory of plant images organized by class subfolders.
    Each class folder may contain multiple subfolders (e.g., collection dates).

    Args:
        models_dict (dict): {model_name: model}
        data_dir (str): path to directory with subdirs (each subdir = class, contains subsubdirs of images)
        class_names (list): list of class names
        display_images (bool): whether to display predictions with matplotlib
        particular_model (str, optional): evaluate only this specific model name
        Minimum (int, optional): skip models with epoch number < Minimum

    Returns:
        results_df (pd.DataFrame): predictions per image
        metrics_df (pd.DataFrame): per-class precision, recall, f1 for each model
    """

    # Transform (same as validation)
    transform = v2.Compose([
        v2.ToImage(),
        v2.Resize(256),
        v2.CenterCrop(224),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406],
                     std=[0.229, 0.224, 0.225])
    ])

    # Storage for all results
    records = []
    metrics_df = pd.DataFrame(columns=["model_name", "plant_species", "precision", "recall", "f1_score"])

    # Iterate over models
    for model_name, model in models_dict.items():
        if particular_model and model_name != particular_model:
            continue

        # Historical optional argument:
        # `Minimum` was retained from an earlier implementation but was not used in the
        # reported Phase 2 workflow. Therefore, the results reported in this study do
        # not depend on the epoch-filtering logic associated with this argument.

        if Minimum and int(model_name[6:8]) < Minimum:
            continue

        device = torch.device("cuda" if torch.cuda.is_available() else "mps")
        model.to(device)
        model.eval()

        all_y_true = []
        all_y_pred = []

        # Walk through dataset
        for class_folder in os.listdir(data_dir):
            class_path = os.path.join(data_dir, class_folder)
            if not os.path.isdir(class_path):
                continue

            # Walk through subfolders inside each class folder
            for root, _, files in os.walk(class_path):
                for img_file in files:
                    if not img_file.lower().endswith(('.png', '.jpg', '.jpeg', '.avif')):
                        continue

                    img_path = os.path.join(root, img_file)

                    # Load and preprocess image
                    image = Image.open(img_path).convert("RGB")
                    input_tensor = transform(image).unsqueeze(0).to(device)

                    # Predict
                    with torch.no_grad():
                        output = model(input_tensor)
                    probs = torch.nn.functional.softmax(output[0], dim=0)
                    confidence, pred_idx = torch.max(probs, 0)

                    predicted_class = class_names[pred_idx.item()]
                    actual_class = class_folder  # main folder name = true label
                    confidence_value = confidence.item() * 100

                    # 🔹 Confidence for the true (actual) class:
                    actual_idx = class_names.index(actual_class)
                    true_confidence_value = probs[actual_idx].item() * 100
                    
                    # Save record
                    records.append({
                        "model_name": model_name,
                        "image_path": img_path,
                        "actual_plant": actual_class,
                        "predicted_plant": predicted_class,
                        "confidence_predicted": confidence_value,
                        "confidence_actual": true_confidence_value
                    })

                    all_y_true.append(class_names.index(actual_class))
                    all_y_pred.append(pred_idx.item())

                    # Optional display
                    if display_images:
                        plt.figure(figsize=(6, 5))
                        plt.imshow(np.array(image))
                        plt.title(f"Actual: {actual_class}\nPredicted: {predicted_class} "
                                  f"({confidence_value:.2f}%)\nEpoch num: {model_name[-2:]}")
                        plt.axis("off")
                        plt.show()

        # Compute per-class metrics
        unique_classes = sorted(set(all_y_true))
        precisions = precision_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0)
        recalls = recall_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0)
        f1s = f1_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0)

        metrics_data = []
        for i, class_idx in enumerate(unique_classes):
            metrics_data.append([
                model_name,
                class_names[class_idx],
                precisions[i],
                recalls[i],
                f1s[i]
            ])

        # Add macro averages
        metrics_data.append([
            model_name,
            "Average Performance (Macro)",
            precisions.mean(),
            recalls.mean(),
            f1s.mean()
        ])

        # Merge metrics
        metrics_df = pd.concat([metrics_df,
                                pd.DataFrame(metrics_data,
                                             columns=["model_name", "plant_species", "precision", "recall", "f1_score"])],
                               ignore_index=True)

    # Final DataFrame for predictions
    results_df = pd.DataFrame(records)

    return results_df, metrics_df


In [16]:
# --- Step 1: Count how many models classified each image correctly ---
def compute_correct_counts(df):
    """
    Compute how many models classified each image correctly,
    total number of models, and the correct ratio.
    """
    correct_counts = (
        df.assign(correct=df['actual_plant'] == df['predicted_plant'])
        .groupby('image_path')['correct']
        .sum()
        .rename('times_correctly_classified')
        .reset_index()
    )

    total_models = df['model_name'].nunique()
    correct_counts['total_models'] = total_models
    correct_counts['correct_ratio'] = correct_counts['times_correctly_classified'] / total_models

    return correct_counts


# --- Step 2: Compute within-model grades ---
def compute_within_model_grades(df):
    df['within_model_grade'] = df.groupby(['model_name', 'actual_plant'])['confidence_actual'] \
                                 .rank(method='dense', ascending=False)
    return df

# --- Step 3: Compute across-model grades ---
def compute_across_model_grades(df):
    mean_conf = df.groupby(['image_path', 'actual_plant'])['confidence_actual'].mean().reset_index()
    mean_conf.rename(columns={'confidence_actual': 'mean_confidence_actual'}, inplace=True)
    mean_conf['across_models_grade'] = mean_conf.groupby('actual_plant')['mean_confidence_actual'] \
                                                .rank(method='dense', ascending=False)
    return df.merge(mean_conf, on=['image_path', 'actual_plant'], how='left')



def assign_percentile(x):
    """
    Assign percentile rank category like 'Top 10%', 'Top 20%', ..., 'Top 100%'.
    Lower percentile number = higher confidence.
    """
    percentile = x.rank(pct=True, ascending=False)
    bins = np.linspace(0, 1, 11)
    labels = [f"Top {int(i*10)}%" for i in range(1, 11)]  # ascending order now
    return pd.cut(percentile, bins=bins, labels=labels, include_lowest=True, right=True)

# --- Step 5: Apply percentile rankings (only for misclassified images) ---
def compute_percentiles(df):
    """
    Compute percentile ranks (within misclassified images only)
    and create a summary column showing both the percentile category
    and how many times each image was correctly classified.
    """
    df = df.copy()

    # Filter misclassified subset (never correctly classified by any model)
    misclassified = df[df['times_correctly_classified'] == 0].copy()

    if misclassified.empty:
        print("✅ No misclassified images (times_correctly_classified == 0) found — skipping percentiles.")
        df['percentile_within_model'] = np.nan
        df['percentile_across_models'] = np.nan
        df['classification_summary'] = (
            "Correctly classified "
            + df['times_correctly_classified'].astype(str)
            + " time(s)"
        )
        return df

    # Compute percentiles within misclassified subset only
    misclassified['percentile_within_model'] = misclassified.groupby(['model_name', 'actual_plant'])['confidence_actual'] \
        .transform(assign_percentile)

    misclassified['percentile_across_models'] = misclassified.groupby('actual_plant')['mean_confidence_actual'] \
        .transform(assign_percentile)

    # Merge back so only misclassified rows have percentiles
    df = df.merge(
        misclassified[['image_path', 'model_name', 'percentile_within_model', 'percentile_across_models']],
        on=['image_path', 'model_name'],
        how='left',
        suffixes=('', '_misclassified')
    )

    # --- New combined summary column ---
    df['classification_summary'] = np.where(
        df['times_correctly_classified'] == 0,
        "Misclassified - Across: " + df['percentile_across_models'].astype(str),
        "Correctly classified " + df['times_correctly_classified'].astype(str) + " time(s)"
    )

    return df




def grade_confidences(results_df):
    """
    Modular pipeline to grade results_df images based on confidence_actual within and across models,
    assign percentile ranks, and record how many models classified each image correctly.

    Adds:
        - within_model_grade
        - across_models_grade
        - mean_confidence_actual
        - percentile_within_model
        - percentile_across_models
        - times_correctly_classified
        - total_models
        - correct_ratio
        - percentile_within_misclassified
    """

    # Step 1 — Compute how many times each image is correctly classified across models
    correct_counts = compute_correct_counts(results_df)
    results_df = results_df.merge(correct_counts, on='image_path', how='left')

    # Step 2 — Compute within-model grading (based on confidence_actual, etc.)
    results_df = compute_within_model_grades(results_df)

    # Step 3 — Compute across-model grading
    results_df = compute_across_model_grades(results_df)

    # Step 4 — Compute percentiles across all data (within and across models)
    results_df = compute_percentiles(results_df)
    
    return results_df

In [17]:
def organize_images_by_performance_group(
    results_df,
    base_output_dir,
    summary_column='classification_summary',
    copy=True
):
    """
    Organize images into 4 main performance-based groups:
        correctly_classified/
        top_30_misclassified/
        mid_30_70_misclassified/
        lowest_30_misclassified/

    Each group contains:
        ClassName / classification_summary / source_name / image.jpg

    Args:
        results_df (pd.DataFrame): Must include ['image_path', 'actual_plant', summary_column]
        base_output_dir (str): Root output directory
        summary_column (str): Column containing summary text (e.g., "Top 10%", "Correctly classified 2 time(s)")
        copy (bool): If True, copy images; if False, move them
    """

    assert summary_column in results_df.columns, f"Column '{summary_column}' not found."

    def categorize(summary_value):
        """Categorize based on classification summary text."""
        text = str(summary_value).lower()
        if "correctly" in text:
            return "correctly_classified"
        match = re.search(r"top\s*(\d+)", text)
        if match:
            rank = int(match.group(1))
            if rank <= 30:
                return "top_30_misclassified"
            elif 30 < rank <= 70:
                return "mid_30_70_misclassified"
            else:
                return "lowest_30_misclassified"
        # fallback if not recognized
        return "unclassified"

    for _, row in results_df.iterrows():
        src_path = row["image_path"]
        class_name = str(row["actual_plant"])
        summary_value = str(row[summary_column])

        if not os.path.exists(src_path):
            print(f"⚠️ Missing file: {src_path}")
            continue

        # Determine category (main folder)
        category = categorize(summary_value)

        # Clean up summary folder name (avoid illegal chars)
        safe_summary = summary_value.replace("/", "-").replace(":", "").replace("*", "-").replace("?", "")
        source_name = os.path.basename(os.path.dirname(src_path))

        # Build final destination path
        dest_dir = os.path.join(base_output_dir, category, class_name, safe_summary, source_name)
        os.makedirs(dest_dir, exist_ok=True)
        dest_path = os.path.join(dest_dir, os.path.basename(src_path))

        try:
            if copy:
                shutil.copy2(src_path, dest_path)
            else:
                shutil.move(src_path, dest_path)
        except Exception as e:
            print(f"❌ Error handling {src_path}: {e}")

    print(f"✅ Images organized into performance groups under: {base_output_dir.name}")


In [21]:
def set_seed(seed: int, deterministic: bool = True):
    """
    Initialize all RNG seeds for reproducibility on Apple Silicon M1.
    Optimized for MPS backend.

    Returns:
        start_rng_state (dict): RNG states captured immediately after seeding.
        generator (torch.Generator): Torch generator initialized with the same seed.
    """
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    if deterministic:
        torch.use_deterministic_algorithms(True)

    generator = torch.Generator()
    generator.manual_seed(seed)

    rng_state = {
        'seed': seed,
        'rng_python': random.getstate(),
        'rng_numpy': np.random.get_state(),
        'rng_torch': torch.get_rng_state(),
        'rng_torch_mps': torch.mps.get_rng_state() if torch.backends.mps.is_available() else None,
        'deterministic': deterministic,
        'generator_state': generator.get_state()
    }
    print(f"✅ Random seed set to {seed} (MPS available: {torch.backends.mps.is_available()})")
    return rng_state, generator

In [22]:
def get_rng_state(seed: int, generator: torch.Generator, deterministic: bool = True):
    """
    Capture the CURRENT RNG states for Python, NumPy, and PyTorch (CPU + MPS if available).

    Returns:
        (dict, torch.Generator): RNG state dict and the generator.
    """
    rng_state = {
        'seed': seed,
        'rng_python': random.getstate(),
        'rng_numpy': np.random.get_state(),
        'rng_torch': torch.get_rng_state(),
        'rng_torch_mps': torch.mps.get_rng_state() if torch.backends.mps.is_available() else None,
        'deterministic': deterministic,
        'generator_state': generator.get_state()
    }
    return rng_state, generator

In [23]:
def manage_rng_state(path: str, mode: str, seed: int = None, generator: torch.Generator = None, deterministic: bool = True):
    """
    Save or load RNG state for Python, NumPy, and PyTorch (CPU + MPS).

    Returns:
        (dict, torch.Generator): RNG state dict and the generator.
    """
    if mode == 'load':
        rng_state = joblib.load(path)

        random.setstate(rng_state['rng_python'])
        np.random.set_state(rng_state['rng_numpy'])
        torch.set_rng_state(rng_state['rng_torch'])
        if torch.backends.mps.is_available() and rng_state.get('rng_torch_mps') is not None:
            torch.mps.set_rng_state(rng_state['rng_torch_mps'])
        torch.use_deterministic_algorithms(rng_state.get('deterministic', False))

        generator = torch.Generator()
        if 'generator_state' in rng_state:
            generator.set_state(rng_state['generator_state'])
        elif seed is not None:
            generator.manual_seed(seed)

        print(f"✅ RNG state loaded from {path}")
        return rng_state, generator

    elif mode == 'save':
        if generator is None:
            generator = torch.Generator().manual_seed(seed)
        rng_state, generator = get_rng_state(seed, generator, deterministic)
        joblib.dump(rng_state, path)
        print(f"✅ RNG state saved to {path}")
        return rng_state, generator

    else:
        raise ValueError("mode must be either 'save' or 'load'")

In [24]:
def setup_reproducibility(start_fresh: bool = False,
                          seed: int = None,
                          deterministic: bool = True,
                          continue_run: bool = False,
                          path: str = None,
                          mode: str = None,
                          generator: torch.Generator = None):
    """
    Wrapper for reproducibility setup:
      - Optionally set RNG seeds.
      - Optionally save/load RNG states.

    Returns:
        (dict, torch.Generator)
    """
    if start_fresh:
        if seed is None:
            raise ValueError("`seed` must be provided when start_fresh=True.")
        return set_seed(seed=seed, deterministic=deterministic)

    if continue_run:
        if path is None or mode is None:
            raise ValueError("Both `path` and `mode` must be provided when continue_run=True.")
        return manage_rng_state(path=path, mode=mode, seed=seed, generator=generator, deterministic=deterministic)

In [25]:
def build_train_transform(
    crop_type="random",
    scale=(0.2, 1.0),
    crop_size=224,
    ratio=(0.75, 1.33),
    color_jitter=True,
    jitter_prob=0.8,
    always_jitter=False,
    hflip_prob=0.5,
    vflip_prob=0.2,
    rotate_prob=0.5,
    rotation_degrees=(-90, 90),
    brightness=(0.7, 1.3),
    contrast=(0.85, 1.3),
    saturation=(0.7, 1.3),
    hue=(-0.0278, 0.0278),
    mean=None,
    std=None,
    normalize=True
):
    """
    Build a training transform pipeline with customizable augmentations.
    ...
    """
    mean = mean or [0.485, 0.456, 0.406]
    std = std or [0.229, 0.224, 0.225]
    
    # Stage 1: Input Preparation & Spatial Augmentations
    transforms_list = [v2.ToImage()]

    if crop_type == "random":
        transforms_list.append(
            v2.RandomResizedCrop(
                size=(crop_size, crop_size),
                scale=scale,
                ratio=ratio,
                antialias=True
            )
        )
    elif crop_type == "center":
        transforms_list.extend([
            v2.Resize(256),
            v2.CenterCrop(crop_size)
        ])
    else:
        raise ValueError(f"Unknown crop_type: {crop_type}. Use 'random' or 'center'")

    transforms_list.extend([
        v2.RandomHorizontalFlip(p=hflip_prob),
        v2.RandomVerticalFlip(p=vflip_prob),
        v2.RandomApply([
            v2.RandomRotation(degrees=rotation_degrees, interpolation=v2.InterpolationMode.BILINEAR)
        ], p=rotate_prob)
    ])

    # Stage 2: Color Operations
    transforms_list.append(v2.ToDtype(torch.float32, scale=True)) # Convert to [0,1] float32

    if color_jitter:
        jitter = v2.ColorJitter(
            brightness=brightness, contrast=contrast,
            saturation=saturation, hue=hue
        )
        transforms_list.append(jitter if always_jitter else v2.RandomApply([jitter], p=jitter_prob)) # Apply color jitter jitter_prob of the time, identity (1 - jitter_prob)

    # Stage 3: Normalization (CRUCIAL for training)
    if normalize:
        transforms_list.append(v2.Normalize(mean=mean, std=std))

    return v2.Compose(transforms_list)

In [26]:
def evaluate_on_three_sets(
    model,
    dict_subset,          # {'Web Top 30': web_top_30_dataset}
    dict_all_web,         # {'All Web Data': web_dataset}
    list_diff_dicts,      # list of dicts, each has one dataset
    transform,
    model_name: str,
    criterion,
    class_names,
    epoch_num,
    log_interval=100,
    batch_size=32,
    num_workers=4
):
    """
    Evaluates model on:
      1. Subset (used in training)
      2. Full web dataset
      3. Each remaining dataset individually
      4. Concatenated remaining datasets
    """

    # Extract keys and datasets
    subset_label, subset_dataset = next(iter(dict_subset.items()))
    all_label, all_dataset = next(iter(dict_all_web.items()))
    subset_dataset.transform = transform
    all_dataset.transform = transform
    

    # Prepare the individual remaining datasets and their names
    remaining_datasets_info = []
    for d in list_diff_dicts:
        label, ds = next(iter(d.items()))
        ds.transform = transform
        remaining_datasets_info.append((ds, label))

    # Concat all remaining datasets together
    concat_remaining = ConcatDataset([ds for ds, _ in remaining_datasets_info])

    # Collect all datasets for evaluation
    datasets_info = [
        (subset_dataset, f"{subset_label} (used in training)"),
        (all_dataset, all_label),
        *remaining_datasets_info,
        (concat_remaining, "Concatenated Remaining Web Data (unseen)")
    ]

    all_results, all_metrics = [], []

    for dataset_obj, dataset_label in datasets_info:
        print(f"\n🔹 Starting evaluation for: {dataset_label}")

        loader = DataLoader(dataset_obj, batch_size=batch_size, shuffle=False, num_workers=num_workers)

        results = evaluate_model(
            model=model,
            val_loader=loader,
            criterion=criterion,
            class_names=class_names,
            epoch_num=epoch_num,
            log_interval=log_interval
        )

        df_results = results['metrics_df'].copy()
        overall_acc = results['overall_accuracy']
        avg_loss = results['avg_loss']
        total_time = results['epoch_time']

        # Add metadata
        df_results.insert(0, "Model", model_name)
        df_results.insert(1, "Dataset", dataset_label)
        all_results.append(df_results)

        # Add summary row
        avg_row = df_results[df_results["Species"] == "Average Performance (Macro)"].copy()
        if not avg_row.empty:
            avg_row = avg_row.iloc[0]
            df_metrics = pd.DataFrame([{
                "Model": model_name,
                "Dataset": dataset_label,
                "Precision": avg_row["Precision"],
                "Recall": avg_row["Recall"],
                "F1-score": avg_row["F1-score"],
                "Epoch": avg_row["Epoch"],
                "overall_accuracy": overall_acc,
                "avg_loss": avg_loss,
                "total_time": total_time
            }])
            all_metrics.append(df_metrics)

    # Combine results
    df_all_results = pd.concat(all_results, ignore_index=True)
    df_metrics_summary = pd.concat(all_metrics, ignore_index=True)

    return df_all_results, df_metrics_summary

# Data Splitting

## External Web Evaluation and Performance Stratification

In [29]:
val_dataset = datasets.ImageFolder(
    root=DATA_DIR / "Validation"  # Path to the validation directory
)

In [30]:
training_states = joblib.load(WORKSPACE_DIR /
                              "Phase 1/mobilenetv2/training_states/training_states_121_130.pkl")

In [31]:
training_states.keys()

dict_keys([121, 122, 123, 124, 125, 126, 127, 128, 129, 130])

In [33]:
model_dict = {}
for i in training_states.keys():
    mobilenet = models.mobilenet_v2(weights=None, num_classes=len(val_dataset.classes)) 
    device = torch.device("mps") 
    mobilenet.to(device)
    mobilenet.load_state_dict(training_states[i]["model_state_dict"])
    model_dict[f'mobilenet_{i}'] = mobilenet

del training_states

In [34]:
model_dict.keys()

dict_keys(['mobilenet_121', 'mobilenet_122', 'mobilenet_123', 'mobilenet_124', 'mobilenet_125', 'mobilenet_126', 'mobilenet_127', 'mobilenet_128', 'mobilenet_129', 'mobilenet_130'])

In [36]:
results_eval_121_130, metrics_eval_121_130 = evaluate_models_on_directory(model_dict,WEB_DATA_DIR, val_dataset.classes)

/var/folders/r9/013lqpnn2cj0hh05j71mb64h0000gn/T/ipykernel_3211/2602539206.py:129: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  metrics_df = pd.concat([metrics_df,


In [37]:
metrics_eval_121_130[metrics_eval_121_130['plant_species']=='Average Performance (Macro)']

,model_name,plant_species,precision,recall,f1_score
32,mobilenet_121,Average Performance (Macro),0.339878,0.205261,0.215530
65,mobilenet_122,Average Performance (Macro),0.334349,0.202834,0.213842
98,mobilenet_123,Average Performance (Macro),0.341831,0.212546,0.221690
131,mobilenet_124,Average Performance (Macro),0.337915,0.217832,0.225721
164,mobilenet_125,Average Performance (Macro),0.331406,0.212019,0.220501
197,mobilenet_126,Average Performance (Macro),0.338396,0.221390,0.231105
230,mobilenet_127,Average Performance (Macro),0.338710,0.226754,0.234675
263,mobilenet_128,Average Performance (Macro),0.344508,0.215790,0.227614
296,mobilenet_129,Average Performance (Macro),0.337081,0.214902,0.223560
329,mobilenet_130,Average Performance (Macro),0.345338,0.214199,0.223350


In [38]:
graded_summary = grade_confidences(results_eval_121_130)

In [70]:
graded_summary.loc[:, graded_summary.columns != "image_path"]

,model_name,actual_plant,predicted_plant,confidence_predicted,confidence_actual,times_correctly_classified,total_models,correct_ratio,within_model_grade,mean_confidence_actual,across_models_grade,percentile_within_model,percentile_across_models,classification_summary
0,mobilenet_121,Ephedra alata Decne.,others,99.785680,0.000235,0,10,0.0,21.0,0.000351,21.0,Top 70%,Top 70%,Misclassified - Across: Top 70%
1,mobilenet_121,Ephedra alata Decne.,others,98.704344,0.000090,0,10,0.0,23.0,0.000234,22.0,Top 80%,Top 70%,Misclassified - Across: Top 70%
2,mobilenet_121,Ephedra alata Decne.,Tamarix nilotica (Ehrenb.) Bunge,75.536013,0.000004,0,10,0.0,24.0,0.000019,24.0,Top 80%,Top 80%,Misclassified - Across: Top 80%
3,mobilenet_121,Ephedra alata Decne.,others,49.295691,1.589664,0,10,0.0,9.0,1.909910,9.0,Top 10%,Top 10%,Misclassified - Across: Top 10%
4,mobilenet_121,Ephedra alata Decne.,others,68.113470,0.000097,0,10,0.0,22.0,0.000176,23.0,Top 70%,Top 80%,Misclassified - Across: Top 80%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58145,mobilenet_130,Salsola imbricata Forssk.,Ochradenus baccatus Delile,68.567693,0.612048,0,10,0.0,109.0,0.433660,131.0,Top 20%,Top 20%,Misclassified - Across: Top 20%
58146,mobilenet_130,Salsola imbricata Forssk.,others,30.679718,15.259188,1,10,0.1,45.0,19.566176,46.0,NaN,NaN,Correctly classified 1 time(s)
58147,mobilenet_130,Salsola imbricata Forssk.,Zygophyllum arabicum (L.) Christenh. & Byng,92.336100,0.000029,0,10,0.0,327.0,0.000081,340.0,Top 70%,Top 70%,Misclassified - Across: Top 70%
58148,mobilenet_130,Salsola imbricata Forssk.,Cebatha pendula (J.R.Forst. & G.Forst.) Kuntze,37.720597,0.000001,0,10,0.0,391.0,0.000003,388.0,Top 90%,Top 90%,Misclassified - Across: Top 90%


In [44]:
graded_summary.to_csv('graded_summary_121_130/graded_summary_121_130.csv')

In [52]:
categorized_web_dir = WEB_DATA_DIR.parent / "categorized_121_130"

In [54]:
organize_images_by_performance_group(graded_summary, categorized_web_dir)

✅ Images organized into performance groups under: categorized_121_130


# **<center>Fine Tuning</center>**

In [49]:
seeds = [42, 1234, 2025]

In [50]:
state, generator = setup_reproducibility(start_fresh=True, seed=seeds[1])

✅ Random seed set to 1234 (MPS available: True)


In [51]:
# Validation/evaluation transforms (no augmentation)
val_transform = v2.Compose([
    v2.ToImage(),
    # Resize the image so its shorter side is 256 pixels, preserving aspect ratio (i.e., the proportion between width and height stays the same)
    v2.Resize(256),
    # Extract a 224x224 patch from the center of the resized image
    v2.CenterCrop(224),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [52]:
# Create FullDataset object.
categorized_web_dir = WEB_DATA_DIR.parent / "categorized_121_130"
Web_correctly_classified = categorized_web_dir / "correctly_classified"
Web_lowest_30_misclassified = categorized_web_dir / "lowest_30_misclassified"
Web_mid_30_70_misclassified = categorized_web_dir / "mid_30_70_misclassified"
Web_top_30_misclassified = categorized_web_dir / "top_30_misclassified"
BATCH_SIZE = 32

In [53]:
train_dataset = datasets.ImageFolder(
    root=TRAIN_DIR
)

val_dataset = datasets.ImageFolder(
    root=VAL_DIR
)

web_dataset = datasets.ImageFolder(
    root=WEB_DATA_DIR
)


web_correctly_classified_dataset = datasets.ImageFolder(
    root=Web_correctly_classified
)


web_lowest_30_dataset = datasets.ImageFolder(
    root=Web_lowest_30_misclassified
)


web_mid_30_70_dataset = datasets.ImageFolder(
    root=Web_mid_30_70_misclassified
)


web_top_30_dataset = datasets.ImageFolder(
    root=Web_top_30_misclassified
)

print(f'train_dataset: {len(train_dataset)}')
print(f'val_dataset: {len(val_dataset)}')
print(f'web_dataset: {len(web_dataset)}')
print(f'web_correctly_classified_dataset: {len(web_correctly_classified_dataset)}')
print(f'web_lowest_30_dataset: {len(web_lowest_30_dataset)}')
print(f'web_mid_30_70_dataset: {len(web_mid_30_70_dataset)}')
print(f'web_top_30_dataset: {len(web_top_30_dataset)}')

train_dataset: 17364
val_dataset: 5200
web_dataset: 5815
web_correctly_classified_dataset: 1815
web_lowest_30_dataset: 1202
web_mid_30_70_dataset: 1599
web_top_30_dataset: 1199


In [54]:
strong_train_transform = build_train_transform(
    scale=(0.08, 1.0),
    jitter_prob=1.0,
    always_jitter=True,
    vflip_prob=0.5,
    rotate_prob=0.7,
    rotation_degrees=(-180, 180),
    normalize=False
                     )
extra_aug = v2.RandomApply([
    v2.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    v2.RandomPerspective(distortion_scale=0.3, p=1.0),
    v2.RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3))
], p=0.5)

new_train_transform = v2.Compose([
    strong_train_transform,
    extra_aug,
    v2.Normalize(mean=[0.485, 0.456, 0.406],
                 std=[0.229, 0.224, 0.225])
])

In [55]:
train_dataset.transform = new_train_transform  
web_lowest_30_dataset.transform = new_train_transform
web_top_30_dataset.transform = new_train_transform
val_dataset.transform = val_transform  

In [61]:
full_web_lowest_30_dataset = ConcatDataset([train_dataset, web_lowest_30_dataset])
full_web_top_30_dataset = ConcatDataset([train_dataset, web_top_30_dataset])

print(f'full_web_lowest_30_dataset: {len(full_web_lowest_30_dataset)}')
print(f'full_web_top_30_dataset: {len(full_web_top_30_dataset)}')

full_web_lowest_30_dataset: 18566
full_web_top_30_dataset: 18563


In [63]:
full_web_lowest_30_loader = DataLoader(
    full_web_lowest_30_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,             # shuffle ensures different order each epoch
    num_workers=4,            # safer on macOS, avoids multiprocessing issues
    pin_memory=True,      
    generator=generator       # ensures reproducible shuffling
)


full_web_top_30_loader = DataLoader(
    full_web_top_30_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,             # shuffle ensures different order each epoch
    num_workers=4,            # safer on macOS, avoids multiprocessing issues
    pin_memory=True,     
    generator=generator       # ensures reproducible shuffling
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,            # no need to shuffle validation
    num_workers=4,            # same note for macOS
    pin_memory=True,
    generator=generator
)

## full_web_top_30_dataset

In [66]:
state, generator = setup_reproducibility(start_fresh=True, seed=seeds[1])

✅ Random seed set to 1234 (MPS available: True)


In [65]:
training_states = joblib.load(WORKSPACE_DIR /
                              "Phase 1/mobilenetv2/training_states/training_states_121_130.pkl")

mobilenetv2_stageF_top_30 = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes)) 
device = torch.device("mps") 
mobilenetv2_stageF_top_30.to(device)
mobilenetv2_stageF_top_30.load_state_dict(training_states[127]["model_state_dict"])
del training_states
df_summary = summarize_mobilenetv2_blocks(mobilenetv2_stageF_top_30)
df_summary

,Block,Order,Module,Layers (#),Params (#),Trainable Params (#),% Layers (Model),% Params (Model),% Trainable Params (Model),Cumulative Layers (Model),Cumulative % Layers (Model),Cumulative Params (Model),Cumulative % Params (Model),Layers (Module %),Params (Module %),Trainable Params (Module %),Cumulative Layers (Module),Cumulative % Layers (Module),Cumulative Params (Module),Cumulative % Params (Module)
0,Conv2dNormActivation_1,1,features,3,928,928,1.898734,0.040951,0.040951,158,100.000000,2266145,100.000000,1.923077,0.041729,0.041729,156,100.000000,2223872,100.000000
1,InvertedResidual_2,2,features,6,896,896,3.797468,0.039539,0.039539,155,98.101266,2265217,99.959049,3.846154,0.040290,0.040290,153,98.076923,2222944,99.958271
2,InvertedResidual_3,3,features,9,5136,5136,5.696203,0.226640,0.226640,149,94.303797,2264321,99.919511,5.769231,0.230949,0.230949,147,94.230769,2222048,99.917981
3,InvertedResidual_4,4,features,9,8832,8832,5.696203,0.389737,0.389737,140,88.607595,2259185,99.692870,5.769231,0.397145,0.397145,138,88.461538,2216912,99.687032
4,InvertedResidual_5,5,features,9,10000,10000,5.696203,0.441278,0.441278,131,82.911392,2250353,99.303134,5.769231,0.449666,0.449666,129,82.692308,2208080,99.289887
5,InvertedResidual_6,6,features,9,14848,14848,5.696203,0.655210,0.655210,122,77.215190,2240353,98.861856,5.769231,0.667664,0.667664,120,76.923077,2198080,98.840221
6,InvertedResidual_7,7,features,9,14848,14848,5.696203,0.655210,0.655210,113,71.518987,2225505,98.206646,5.769231,0.667664,0.667664,111,71.153846,2183232,98.172557
7,InvertedResidual_8,8,features,9,21056,21056,5.696203,0.929155,0.929155,104,65.822785,2210657,97.551436,5.769231,0.946817,0.946817,102,65.384615,2168384,97.504892
8,InvertedResidual_9,9,features,9,54272,54272,5.696203,2.394904,2.394904,95,60.126582,2189601,96.622281,5.769231,2.440428,2.440428,93,59.615385,2147328,96.558075
9,InvertedResidual_10,10,features,9,54272,54272,5.696203,2.394904,2.394904,86,54.430380,2135329,94.227377,5.769231,2.440428,2.440428,84,53.846154,2093056,94.117647


### the following method divide the other class weights by 2

In [98]:
norm_sqrt_inv_weights_train = compute_class_weights(train_dataset, method="sqrt_inverse", normalize=True)
norm_sqrt_inv_weights_train_top_30 = compute_class_weights(web_top_30_dataset, method="sqrt_inverse", normalize=True)   
norm_sqrt_inv_weights_train_top_30[-1] = 0 # handel other class

In [99]:
norm_sqrt_inv_weights = (norm_sqrt_inv_weights_train + (2*norm_sqrt_inv_weights_train_top_30))/2

In [100]:
norm_sqrt_inv_weights

tensor([1.0938, 1.1379, 1.5019, 1.2137, 1.1484, 1.2588, 1.5647, 1.8985, 2.4456,
        1.0494, 1.6496, 2.0818, 1.0580, 0.9910, 1.0173, 1.4435, 2.7015, 1.5595,
        1.1196, 0.9252, 0.9630, 1.5881, 1.1269, 0.8456, 1.2772, 1.0728, 1.4412,
        1.1018, 0.8796, 1.2396, 1.7300, 2.5961, 0.4783], device='mps:0')

In [101]:
norm_sqrt_inv_criterion = nn.CrossEntropyLoss(weight=norm_sqrt_inv_weights)

In [ ]:
stageF_optimizer = torch.optim.AdamW(mobilenetv2_stageF_top_30.parameters(), lr=1e-5,
                            weight_decay=1e-4, betas=(0.9, 0.999))

In [42]:
start_epoch = 128
end_epoch = 140
start_epoch, end_epoch

(128, 140)

In [43]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_top_30, full_web_top_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_top_30', save_dir = 'Training outputs',
                         class_weights='norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 128 - Training Started...

Epoch 128, Batch 100, Loss: 0.6012, Accuracy: 86.66%, Time Passed: 3.57m
Epoch 128, Batch 200, Loss: 0.5890, Accuracy: 86.59%, Time Passed: 7.11m
Epoch 128, Batch 300, Loss: 0.5908, Accuracy: 86.76%, Time Passed: 10.74m
Epoch 128, Batch 400, Loss: 0.5675, Accuracy: 87.01%, Time Passed: 14.57m
Epoch 128, Batch 500, Loss: 0.5503, Accuracy: 87.29%, Time Passed: 18.34m
Epoch 128 Completed - Average Loss: 0.5459, Accuracy: 87.32%, Epoch Time: 21.30m
Training Complete Epoch 128 - Total Time: 21.30m

✅ Epoch 128 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.7834, Accuracy: 80.19%, Time Passed: 4.44m
Evaluation Complete - Loss: 0.7482, Accuracy: 82.53%, Total Time: 8.12m

📊 Epoch 128 - Evaluation Completed.


🚀 Epoch 129 - Training Started...

Epoch 129, Batch 100, Loss: 0.4194, Accuracy: 88.59%, Time Passed: 4.36m
Epoch 129, Batch 200, Loss: 0.4345, Accuracy: 88.25%, Time Passed: 8.91m
Epoch 129, Batch 300, Loss: 0.4361, Accuracy: 88.29%,

In [44]:
if training_states:
    joblib.dump(training_states, f'Training outputs/training_states_top_30_{start_epoch}_{end_epoch}.pkl')

In [45]:
del training_states

### **Continue Training**

In [102]:
state, generator = setup_reproducibility(continue_run=True, path="Training outputs/rng_state_140.pkl", mode="load")

✅ RNG state loaded from Training outputs/rng_state_140.pkl


> **Output-organization note:** Training outputs were initially generated in the main `Training outputs/` directory and were subsequently organized manually into dedicated subdirectories such as `training_states/`, `top_val_results/`, and `lowest_val_results/`. These paths therefore reflect the organized archive structure rather than a different training procedure.

In [103]:
training_states = joblib.load('Training outputs/training_states_top_30_128_140.pkl')

# Recreate the optimizer with the same settings as before
stageF_optimizer = torch.optim.AdamW(mobilenetv2_stageF_top_30.parameters(), lr=1e-5,
                            weight_decay=1e-4, betas=(0.9, 0.999))

# Load the saved optimizer state
stageF_optimizer.load_state_dict(training_states[140]["optimizer_state_dict"])
del training_states

In [104]:
start_epoch = 141
end_epoch = 150
start_epoch, end_epoch

(141, 150)

In [71]:
val_res = evaluate_model(mobilenetv2_stageF_top_30, val_loader, norm_sqrt_inv_criterion, train_dataset.classes, epoch_num=1, log_interval=100)

python(40604) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(40606) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(40608) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(40610) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Batch 100, Loss: 0.6091, Accuracy: 83.16%, Time Passed: 3.49m
Evaluation Complete - Loss: 0.5958, Accuracy: 85.03%, Total Time: 6.55m


In [105]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_top_30, full_web_top_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_top_30', save_dir = 'Training outputs',
                         class_weights='norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 141 - Training Started...



python(40782) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(40784) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(40787) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(40789) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Epoch 141, Batch 100, Loss: 0.2433, Accuracy: 91.66%, Time Passed: 4.58m
Epoch 141, Batch 200, Loss: 0.2445, Accuracy: 91.86%, Time Passed: 8.83m
Epoch 141, Batch 300, Loss: 0.2625, Accuracy: 91.54%, Time Passed: 13.37m
Epoch 141, Batch 400, Loss: 0.2600, Accuracy: 91.77%, Time Passed: 18.03m
Epoch 141, Batch 500, Loss: 0.2651, Accuracy: 91.54%, Time Passed: 22.52m
Epoch 141 Completed - Average Loss: 0.2656, Accuracy: 91.53%, Epoch Time: 25.98m
Training Complete Epoch 141 - Total Time: 25.98m

✅ Epoch 141 - Training Completed. Starting Evaluation...



python(41167) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(41169) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(41171) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(41173) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Batch 100, Loss: 0.5728, Accuracy: 84.69%, Time Passed: 4.06m
Evaluation Complete - Loss: 0.5728, Accuracy: 86.01%, Total Time: 7.37m

📊 Epoch 141 - Evaluation Completed.


🚀 Epoch 142 - Training Started...



...

Batch 100, Loss: 0.5378, Accuracy: 85.50%, Time Passed: 4.08m
Evaluation Complete - Loss: 0.5525, Accuracy: 86.27%, Total Time: 7.44m

📊 Epoch 149 - Evaluation Completed.


🚀 Epoch 150 - Training Started...



python(44618) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(44620) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(44622) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(44624) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Epoch 150, Batch 100, Loss: 0.2303, Accuracy: 92.81%, Time Passed: 4.88m
Epoch 150, Batch 200, Loss: 0.2339, Accuracy: 92.70%, Time Passed: 9.16m
Epoch 150, Batch 300, Loss: 0.2366, Accuracy: 92.51%, Time Passed: 13.86m
Epoch 150, Batch 400, Loss: 0.2354, Accuracy: 92.59%, Time Passed: 18.48m
Epoch 150, Batch 500, Loss: 0.2360, Accuracy: 92.62%, Time Passed: 24.04m
Epoch 150 Completed - Average Loss: 0.2328, Accuracy: 92.64%, Epoch Time: 27.80m
Training Complete Epoch 150 - Total Time: 27.80m

✅ Epoch 150 - Training Completed. Starting Evaluation...



python(44931) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(44934) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(44936) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(44939) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Batch 100, Loss: 0.5659, Accuracy: 84.19%, Time Passed: 4.17m
Evaluation Complete - Loss: 0.5565, Accuracy: 85.79%, Total Time: 7.70m

📊 Epoch 150 - Evaluation Completed.



In [107]:
if training_states:
    joblib.dump(training_states, f'Training outputs/training_states_top_30_{start_epoch}_{end_epoch}.pkl')

In [108]:
del training_states

In [109]:
state, generator = setup_reproducibility(continue_run=True, path="Training outputs/rng_state_150.pkl", mode="save", seed=seeds[1], generator=generator)

✅ RNG state saved to Training outputs/rng_state_150.pkl


In [49]:
training_states = joblib.load("Training outputs/training_states_top_30_141_150.pkl"
)
mobilenetv2_stageF_top_30 = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes)) 
device = torch.device("mps") 
mobilenetv2_stageF_top_30.to(device)
mobilenetv2_stageF_top_30.load_state_dict(training_states[150]["model_state_dict"])
del training_states

norm_sqrt_inv_weights_train = compute_class_weights(train_dataset, method="sqrt_inverse", normalize=True)
norm_sqrt_inv_weights_train_top_30 = compute_class_weights(web_top_30_dataset, method="sqrt_inverse", normalize=True)   
norm_sqrt_inv_weights_train_top_30[-1] = 0 # handel other class
norm_sqrt_inv_weights = (norm_sqrt_inv_weights_train + (2*norm_sqrt_inv_weights_train_top_30))/2
norm_sqrt_inv_criterion = nn.CrossEntropyLoss(weight=norm_sqrt_inv_weights)

web_datasets_dict_list = [{'All Web Data': web_dataset},
 {'Web Correctly Classified': web_correctly_classified_dataset},
 {'Web Top 30': web_top_30_dataset}, 
 {'Web Mid_30_70': web_mid_30_70_dataset},
 {'Web Lowest 30': web_lowest_30_dataset}
]

> **Evaluation-set note:** The label `unseen` is retained from the historical code. It means that these web images were not used for training the current branch; however, they belong to the broader web pool previously used for performance stratification and should not be interpreted as an independent unseen test set.

In [50]:
top_30_all_results, top_30_metrics_summary = evaluate_on_three_sets(
    mobilenetv2_stageF_top_30,
    web_datasets_dict_list[2],          # {'Web Top 30': web_top_30_dataset}
    web_datasets_dict_list[0],         # {'All Web Data': web_dataset}
    [d for i, d in enumerate(web_datasets_dict_list) if i not in (0, 2)],      # list of dicts, each has one dataset
    val_transform,
    "mobilenetv2_stageF_top_30",
    norm_sqrt_inv_criterion,
    train_dataset.classes,
    epoch_num=150,
    log_interval=100,
    batch_size=32,
    num_workers=4
)


🔹 Starting evaluation for: Web Top 30 (used in training)
Evaluation Complete - Loss: 0.5333, Accuracy: 80.08%, Total Time: 0.55m

🔹 Starting evaluation for: All Web Data
Batch 100, Loss: 2.0355, Accuracy: 56.38%, Time Passed: 0.37m
Evaluation Complete - Loss: 2.1240, Accuracy: 58.32%, Total Time: 0.93m

🔹 Starting evaluation for: Web Correctly Classified
Evaluation Complete - Loss: 9.4798, Accuracy: 31.02%, Total Time: 0.60m

🔹 Starting evaluation for: Web Mid_30_70
Evaluation Complete - Loss: 2.4290, Accuracy: 41.38%, Total Time: 0.58m

🔹 Starting evaluation for: Web Lowest 30
Evaluation Complete - Loss: 5.9142, Accuracy: 16.71%, Total Time: 0.55m

🔹 Starting evaluation for: Concatenated Remaining Web Data (unseen)
Batch 100, Loss: 6.4583, Accuracy: 35.25%, Time Passed: 0.37m
Evaluation Complete - Loss: 6.1220, Accuracy: 30.88%, Total Time: 0.84m


In [78]:
## Note on legacy filenames
## Some historical output filenames use the term test. These files contain external web evaluation results and do not represent an independent test set.

In [141]:
top_30_all_results.to_csv("Training outputs/top_30_web_test_150_results.csv", index=False)
top_30_metrics_summary.to_csv("Training outputs/top_30_web_test_150_metrics.csv", index=False)

### Continue training 

In [100]:
state, generator = setup_reproducibility(start_fresh=True, seed=seeds[2])

✅ Random seed set to 2025 (MPS available: True)


In [101]:
training_states = joblib.load("Training outputs/training_states_top_30_141_150.pkl")
mobilenetv2_stageF_top_30 = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes)) 
device = torch.device("mps") 
mobilenetv2_stageF_top_30.to(device)
mobilenetv2_stageF_top_30.load_state_dict(training_states[150]["model_state_dict"])
del training_states

In [146]:
norm_sqrt_inv_weights_train = compute_class_weights(train_dataset, method="sqrt_inverse", normalize=True)
norm_sqrt_inv_weights_train_top_30 = compute_class_weights(web_top_30_dataset, method="sqrt_inverse", normalize=True)   
norm_sqrt_inv_weights_train_top_30[-1] = 0 # handel other class
norm_sqrt_inv_weights = (norm_sqrt_inv_weights_train + (2*norm_sqrt_inv_weights_train_top_30))/2

top_30_val_results141_150 = pd.read_csv("Training outputs/mobilenetv2_stageF_top_30_val_results141_150.csv")

# Step 1: get the subset dataframe
df = top_30_val_results141_150[
    (top_30_val_results141_150['Species'] != "Average Performance (Macro)") &
    (top_30_val_results141_150['Epoch'] == 150)
].copy()

# Step 2: build a mapping from species to recall multiplier
def get_multiplier(recall):
    if recall < 0.6:
        return 10
    elif recall < 0.7:
        return 8
    elif recall < 0.8:
        return 6
    elif recall < 0.9:
        return 4
    elif recall < 1:
        return 2
    else:
        return 1

df['Multiplier'] = df['Recall'].apply(get_multiplier)

# Step 3: make sure class order matches your dataset order
species_to_multiplier = dict(zip(df['Species'], df['Multiplier']))

# Step 4: apply the multipliers to your weight vector
adjusted_weights = norm_sqrt_inv_weights.clone() if hasattr(norm_sqrt_inv_weights, 'clone') else np.copy(norm_sqrt_inv_weights)

for idx, class_name in enumerate(train_dataset.classes):
    if class_name in species_to_multiplier:
        adjusted_weights[idx] *= species_to_multiplier[class_name]

In [147]:
df

,Species,Precision,Recall,F1-score,Epoch,Multiplier
306,Anabasis articulata (Forssk.) Moq.,0.861111,1.000000,0.925373,150,1
307,Anabasis setifera Moq.,0.897590,0.752525,0.818681,150,6
308,Atriplex halimus L.,0.688202,0.871886,0.769231,150,4
309,Calotropis procera,0.960526,0.901235,0.929936,150,2
310,Capparis spinosa L.,0.947761,0.894366,0.920290,150,4
311,Cebatha pendula (J.R.Forst. & G.Forst.) Kuntze,0.931373,0.791667,0.855856,150,6
312,Cenchrus divisus,0.977273,0.988506,0.982857,150,2
313,Deverra tortuosa (Desf.) DC.,0.802326,0.985714,0.884615,150,2
314,Deverra triradiata Hochst. ex Boiss.,0.704225,1.000000,0.826446,150,1
315,Diplotaxis harra (Forssk.) Boiss.,0.890511,0.897059,0.893773,150,4


In [148]:
adjusted_weights, norm_sqrt_inv_weights

(tensor([ 1.0938,  6.8274,  6.0075,  2.4275,  4.5935,  7.5528,  3.1295,  3.7970,
          2.4456,  4.1975,  3.2993,  8.3270,  8.4638,  1.9820,  4.0691,  2.8870,
          2.7015,  9.3570,  8.9568,  5.5509,  1.9259,  1.5881,  1.1269,  0.8456,
          1.2772,  2.1456, 11.5294, 11.0180,  0.8796,  2.4793,  3.4599,  5.1923,
          1.9130], device='mps:0'),
 tensor([1.0938, 1.1379, 1.5019, 1.2137, 1.1484, 1.2588, 1.5647, 1.8985, 2.4456,
         1.0494, 1.6496, 2.0818, 1.0580, 0.9910, 1.0173, 1.4435, 2.7015, 1.5595,
         1.1196, 0.9252, 0.9630, 1.5881, 1.1269, 0.8456, 1.2772, 1.0728, 1.4412,
         1.1018, 0.8796, 1.2396, 1.7300, 2.5961, 0.4783], device='mps:0'))

In [149]:
norm_sqrt_inv_criterion = nn.CrossEntropyLoss(weight=adjusted_weights)
stageF_optimizer = torch.optim.AdamW(mobilenetv2_stageF_top_30.parameters(), lr=1e-5,
                            weight_decay=1e-4, betas=(0.9, 0.999))

In [106]:
start_epoch = 151
end_epoch = 155
start_epoch, end_epoch

(151, 155)

In [107]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_top_30, full_web_top_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_top_30', save_dir = 'Training outputs',
                         class_weights='adjusted_norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 151 - Training Started...

Epoch 151, Batch 100, Loss: 0.2451, Accuracy: 92.22%, Time Passed: 3.75m
Epoch 151, Batch 200, Loss: 0.2274, Accuracy: 92.53%, Time Passed: 7.74m
Epoch 151, Batch 300, Loss: 0.2188, Accuracy: 92.69%, Time Passed: 11.52m
Epoch 151, Batch 400, Loss: 0.2180, Accuracy: 92.68%, Time Passed: 15.44m
Epoch 151, Batch 500, Loss: 0.2147, Accuracy: 92.73%, Time Passed: 19.20m
Epoch 151 Completed - Average Loss: 0.2141, Accuracy: 92.69%, Epoch Time: 21.88m
Training Complete Epoch 151 - Total Time: 21.88m

✅ Epoch 151 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.5198, Accuracy: 85.91%, Time Passed: 3.95m
Evaluation Complete - Loss: 0.5439, Accuracy: 86.79%, Total Time: 7.11m

📊 Epoch 151 - Evaluation Completed.


🚀 Epoch 152 - Training Started...

Epoch 152, Batch 100, Loss: 0.2054, Accuracy: 92.69%, Time Passed: 3.57m
Epoch 152, Batch 200, Loss: 0.1949, Accuracy: 92.38%, Time Passed: 7.24m
Epoch 152, Batch 300, Loss: 0.1920, Accuracy: 92.59%,

In [110]:
if training_states:
    joblib.dump(training_states, f'Training outputs/training_states_top_30_{start_epoch}_{end_epoch}.pkl')

In [139]:
top_30_all_results_155, top_30_metrics_summary_155 = evaluate_on_three_sets(
    mobilenetv2_stageF_top_30,
    web_datasets_dict_list[2],          # {'Web Top 30': web_top_30_dataset}
    web_datasets_dict_list[0],         # {'All Web Data': web_dataset}
    [d for i, d in enumerate(web_datasets_dict_list) if i not in (0, 2)],      # list of dicts, each has one dataset
    val_transform,
    "mobilenetv2_stageF_top_30",
    norm_sqrt_inv_criterion,
    train_dataset.classes,
    epoch_num=155,
    log_interval=100,
    batch_size=32,
    num_workers=4
)


🔹 Starting evaluation for: Web Top 30 (used in training)


python(10356) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10358) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10360) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10362) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Evaluation Complete - Loss: 0.5421, Accuracy: 80.00%, Total Time: 0.69m

🔹 Starting evaluation for: All Web Data


python(10405) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10407) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10409) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10411) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Batch 100, Loss: 2.0106, Accuracy: 57.47%, Time Passed: 0.46m
Evaluation Complete - Loss: 2.2323, Accuracy: 56.89%, Total Time: 1.04m

🔹 Starting evaluation for: Web Correctly Classified


...

Evaluation Complete - Loss: 2.5768, Accuracy: 37.75%, Total Time: 0.68m

🔹 Starting evaluation for: Web Lowest 30


python(10467) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10469) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10471) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10476) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Evaluation Complete - Loss: 6.2854, Accuracy: 12.64%, Total Time: 0.64m

🔹 Starting evaluation for: Concatenated Remaining Web Data (unseen)


python(10496) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10499) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10501) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10503) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Batch 100, Loss: 6.5327, Accuracy: 33.84%, Time Passed: 0.53m
Evaluation Complete - Loss: 6.2519, Accuracy: 28.71%, Total Time: 1.01m


In [142]:
top_30_all_results_155.to_csv("Training outputs/top_30_web_test_155_results.csv", index=False)
top_30_metrics_summary_155.to_csv("Training outputs/top_30_web_test_155_metrics.csv", index=False)

In [150]:
val_res = evaluate_model(mobilenetv2_stageF_top_30, val_loader, norm_sqrt_inv_criterion, train_dataset.classes, epoch_num=155, log_interval=100)

python(10820) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10822) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10824) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10826) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Batch 100, Loss: 0.4704, Accuracy: 86.97%, Time Passed: 5.75m
Evaluation Complete - Loss: 0.4945, Accuracy: 87.59%, Total Time: 9.85m


In [151]:
start_epoch = 156
end_epoch = 160
start_epoch, end_epoch

(156, 160)

In [152]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_top_30, full_web_top_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_top_30', save_dir = 'Training outputs',
                         class_weights='adjusted_norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 156 - Training Started...



python(10913) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10916) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10918) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(10920) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Epoch 156, Batch 100, Loss: 0.1983, Accuracy: 92.44%, Time Passed: 6.93m
Epoch 156, Batch 200, Loss: 0.1813, Accuracy: 92.95%, Time Passed: 15.27m
Epoch 156, Batch 300, Loss: 0.1817, Accuracy: 92.89%, Time Passed: 28.35m
Epoch 156, Batch 400, Loss: 0.1778, Accuracy: 92.83%, Time Passed: 36.50m
Epoch 156, Batch 500, Loss: 0.1760, Accuracy: 92.86%, Time Passed: 42.62m
Epoch 156 Completed - Average Loss: 0.1784, Accuracy: 92.79%, Epoch Time: 46.08m
Training Complete Epoch 156 - Total Time: 46.08m

✅ Epoch 156 - Training Completed. Starting Evaluation...



python(11965) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(11968) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(11970) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(11972) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Batch 100, Loss: 0.4829, Accuracy: 86.91%, Time Passed: 4.25m
Evaluation Complete - Loss: 0.5030, Accuracy: 87.43%, Total Time: 7.83m

📊 Epoch 156 - Evaluation Completed.


🚀 Epoch 157 - Training Started...



...

Batch 100, Loss: 0.4711, Accuracy: 86.72%, Time Passed: 4.56m
Evaluation Complete - Loss: 0.4798, Accuracy: 87.56%, Total Time: 8.17m

📊 Epoch 159 - Evaluation Completed.


🚀 Epoch 160 - Training Started...



python(13360) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(13362) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(13364) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(13366) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Epoch 160, Batch 100, Loss: 0.1915, Accuracy: 92.53%, Time Passed: 5.46m
Epoch 160, Batch 200, Loss: 0.1821, Accuracy: 92.84%, Time Passed: 10.73m
Epoch 160, Batch 300, Loss: 0.1871, Accuracy: 92.60%, Time Passed: 15.61m
Epoch 160, Batch 400, Loss: 0.1866, Accuracy: 92.45%, Time Passed: 21.08m
Epoch 160, Batch 500, Loss: 0.1885, Accuracy: 92.35%, Time Passed: 26.01m
Epoch 160 Completed - Average Loss: 0.1867, Accuracy: 92.44%, Epoch Time: 29.83m
Training Complete Epoch 160 - Total Time: 29.83m

✅ Epoch 160 - Training Completed. Starting Evaluation...



python(13953) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(13955) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(13958) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(13961) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Batch 100, Loss: 0.4707, Accuracy: 86.78%, Time Passed: 4.88m
Evaluation Complete - Loss: 0.4880, Accuracy: 87.44%, Total Time: 8.80m

📊 Epoch 160 - Evaluation Completed.



In [153]:
if training_states:
    joblib.dump(training_states, f'Training outputs/training_states_top_30_{start_epoch}_{end_epoch}.pkl')

### Continue training

In [57]:
state, generator = setup_reproducibility(start_fresh=True, seed=seeds[0])

✅ Random seed set to 42 (MPS available: True)


In [58]:
training_states = joblib.load("Training outputs/training_states_top_30_156_160.pkl")
mobilenetv2_stageF_top_30 = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes)) 
device = torch.device("mps") 
mobilenetv2_stageF_top_30.to(device)
mobilenetv2_stageF_top_30.load_state_dict(training_states[160]["model_state_dict"])
del training_states

- **Note on validation-output organization**
- Validation CSV files generated during training were subsequently organized manually into the top_val_results/ and lowest_val_results/ subdirectories. These - paths therefore refer to the organized output structure rather than the original save location.

In [146]:
norm_sqrt_inv_weights_train = compute_class_weights(train_dataset, method="sqrt_inverse", normalize=True)
norm_sqrt_inv_weights_train_top_30 = compute_class_weights(web_top_30_dataset, method="sqrt_inverse", normalize=True)   
norm_sqrt_inv_weights_train_top_30[-1] = 0 # handel other class
norm_sqrt_inv_weights = (norm_sqrt_inv_weights_train + (2*norm_sqrt_inv_weights_train_top_30))/2

top_30_val_results156_160 = pd.read_csv("Training outputs/top_val_results/mobilenetv2_stageF_top_30_val_results156_160.csv")

# Step 1: get the subset dataframe
df = top_30_val_results156_160[
    (top_30_val_results156_160['Species'] != "Average Performance (Macro)") &
    (top_30_val_results156_160['Epoch'] == 160)
].copy()

# Step 2: build a mapping from species to recall multiplier
def get_multiplier(recall):
    if recall < 0.6:
        return 10
    elif recall < 0.7:
        return 8
    elif recall < 0.8:
        return 6
    elif recall < 0.9:
        return 4
    elif recall < 1:
        return 2
    else:
        return 1

df['Multiplier'] = df['Recall'].apply(get_multiplier)

# Step 3: make sure class order matches your dataset order
species_to_multiplier = dict(zip(df['Species'], df['Multiplier']))

# Step 4: apply the multipliers to your weight vector
adjusted_weights = norm_sqrt_inv_weights.clone() if hasattr(norm_sqrt_inv_weights, 'clone') else np.copy(norm_sqrt_inv_weights)

for idx, class_name in enumerate(train_dataset.classes):
    if class_name in species_to_multiplier:
        adjusted_weights[idx] *= species_to_multiplier[class_name]

In [60]:
df

,Species,Precision,Recall,F1-score,Epoch,Multiplier
136,Anabasis articulata (Forssk.) Moq.,0.953125,0.983871,0.968254,160,2
137,Anabasis setifera Moq.,0.881720,0.828283,0.854167,160,4
138,Atriplex halimus L.,0.753994,0.839858,0.794613,160,4
139,Calotropis procera,0.961039,0.913580,0.936709,160,2
140,Capparis spinosa L.,0.961240,0.873239,0.915129,160,4
141,Cebatha pendula (J.R.Forst. & G.Forst.) Kuntze,0.921569,0.783333,0.846847,160,6
142,Cenchrus divisus,0.988636,1.000000,0.994286,160,1
143,Deverra tortuosa (Desf.) DC.,0.777778,1.000000,0.875000,160,1
144,Deverra triradiata Hochst. ex Boiss.,0.859649,0.980000,0.915888,160,2
145,Diplotaxis harra (Forssk.) Boiss.,0.808917,0.933824,0.866894,160,2


In [147]:
adjusted_weights, norm_sqrt_inv_weights

(tensor([ 2.1876,  4.5516,  6.0075,  2.4275,  4.5935,  7.5528,  1.5647,  1.8985,
          4.8911,  2.0988,  3.2993,  4.1635,  6.3478,  1.9820,  2.0345,  5.7740,
          2.7015,  9.3570,  6.7176,  3.7006,  1.9259,  1.5881,  1.1269,  0.8456,
          1.2772,  2.1456, 11.5294, 11.0180,  0.8796,  2.4793,  3.4599,  5.1923,
          1.9130], device='mps:0'),
 tensor([1.0938, 1.1379, 1.5019, 1.2137, 1.1484, 1.2588, 1.5647, 1.8985, 2.4456,
         1.0494, 1.6496, 2.0818, 1.0580, 0.9910, 1.0173, 1.4435, 2.7015, 1.5595,
         1.1196, 0.9252, 0.9630, 1.5881, 1.1269, 0.8456, 1.2772, 1.0728, 1.4412,
         1.1018, 0.8796, 1.2396, 1.7300, 2.5961, 0.4783], device='mps:0'))

In [148]:
norm_sqrt_inv_criterion = nn.CrossEntropyLoss(weight=adjusted_weights)
stageF_optimizer = torch.optim.AdamW(mobilenetv2_stageF_top_30.parameters(), lr=1e-5,
                            weight_decay=1e-4, betas=(0.9, 0.999))

In [63]:
start_epoch = 161
end_epoch = 180
start_epoch, end_epoch

(161, 180)

In [64]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_top_30, full_web_top_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_top_30', save_dir = 'Training outputs',
                         class_weights='adjusted_norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 161 - Training Started...

Epoch 161, Batch 100, Loss: 0.1827, Accuracy: 92.44%, Time Passed: 3.84m
Epoch 161, Batch 200, Loss: 0.1880, Accuracy: 92.20%, Time Passed: 7.65m
Epoch 161, Batch 300, Loss: 0.1812, Accuracy: 92.46%, Time Passed: 11.22m
Epoch 161, Batch 400, Loss: 0.1819, Accuracy: 92.34%, Time Passed: 14.87m
Epoch 161, Batch 500, Loss: 0.1781, Accuracy: 92.58%, Time Passed: 18.64m
Epoch 161 Completed - Average Loss: 0.1777, Accuracy: 92.70%, Epoch Time: 21.53m
Training Complete Epoch 161 - Total Time: 21.53m

✅ Epoch 161 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.4824, Accuracy: 86.84%, Time Passed: 4.33m
Evaluation Complete - Loss: 0.5045, Accuracy: 87.33%, Total Time: 7.51m

📊 Epoch 161 - Evaluation Completed.


🚀 Epoch 162 - Training Started...

Epoch 162, Batch 100, Loss: 0.1700, Accuracy: 93.84%, Time Passed: 3.79m
Epoch 162, Batch 200, Loss: 0.1707, Accuracy: 93.38%, Time Passed: 7.34m
Epoch 162, Batch 300, Loss: 0.1707, Accuracy: 93.18%,

In [66]:
if training_states:
    joblib.dump(training_states, f'Training outputs/training_states_top_30_{start_epoch}_{end_epoch}.pkl')

In [72]:
web_datasets_dict_list = [{'All Web Data': web_dataset},
 {'Web Correctly Classified': web_correctly_classified_dataset},
 {'Web Top 30': web_top_30_dataset}, 
 {'Web Mid_30_70': web_mid_30_70_dataset},
 {'Web Lowest 30': web_lowest_30_dataset}
]
top_30_all_results_180, top_30_metrics_summary_180 = evaluate_on_three_sets(
    mobilenetv2_stageF_top_30,
    web_datasets_dict_list[2],          # {'Web Top 30': web_top_30_dataset}
    web_datasets_dict_list[0],         # {'All Web Data': web_dataset}
    [d for i, d in enumerate(web_datasets_dict_list) if i not in (0, 2)],      # list of dicts, each has one dataset
    val_transform,
    "mobilenetv2_stageF_top_30",
    norm_sqrt_inv_criterion,
    train_dataset.classes,
    epoch_num=180,
    log_interval=100,
    batch_size=32,
    num_workers=4
)


🔹 Starting evaluation for: Web Top 30 (used in training)
Evaluation Complete - Loss: 0.2998, Accuracy: 89.75%, Total Time: 0.74m

🔹 Starting evaluation for: All Web Data
Batch 100, Loss: 1.7287, Accuracy: 63.28%, Time Passed: 0.41m
Evaluation Complete - Loss: 1.9914, Accuracy: 62.07%, Total Time: 0.99m

🔹 Starting evaluation for: Web Correctly Classified
Evaluation Complete - Loss: 9.8288, Accuracy: 31.13%, Total Time: 0.64m

🔹 Starting evaluation for: Web Mid_30_70
Evaluation Complete - Loss: 2.2675, Accuracy: 45.62%, Total Time: 0.64m

🔹 Starting evaluation for: Web Lowest 30
Evaluation Complete - Loss: 5.7182, Accuracy: 17.46%, Total Time: 0.58m

🔹 Starting evaluation for: Concatenated Remaining Web Data (unseen)
Batch 100, Loss: 6.5728, Accuracy: 37.12%, Time Passed: 0.40m
Evaluation Complete - Loss: 6.1378, Accuracy: 32.59%, Total Time: 0.87m


In [73]:
top_30_all_results_180.to_csv("Training outputs/top_30_web_test_180_results.csv", index=False)
top_30_metrics_summary_180.to_csv("Training outputs/top_30_web_test_180_metrics.csv", index=False)

### New Method 

In [50]:
state, generator = setup_reproducibility(start_fresh=True, seed=seeds[2])

✅ Random seed set to 2025 (MPS available: True)


In [51]:
training_states = joblib.load("Training outputs/training_states/training_states_top_30_161_180.pkl")
mobilenetv2_stageF_top_30 = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes)) 
device = torch.device("mps") 
mobilenetv2_stageF_top_30.to(device)
mobilenetv2_stageF_top_30.load_state_dict(training_states[180]["model_state_dict"])
del training_states

In [52]:
default_train_transform = build_train_transform(crop_type="random", scale=(0.2, 1.0))
train_dataset.transform = default_train_transform  
web_top_30_dataset.transform = default_train_transform
val_dataset.transform = val_transform  
full_web_top_30_dataset = ConcatDataset([train_dataset, web_top_30_dataset])
train_dataset.transform

Compose(
      ToImage()
      RandomResizedCrop(size=(224, 224), scale=(0.2, 1.0), ratio=(0.75, 1.33), interpolation=InterpolationMode.BILINEAR, antialias=True)
      RandomHorizontalFlip(p=0.5)
      RandomVerticalFlip(p=0.2)
      RandomApply(    RandomRotation(degrees=[-90.0, 90.0], interpolation=InterpolationMode.BILINEAR, expand=False, fill=0))
      ToDtype(scale=True)
      RandomApply(    ColorJitter(brightness=(0.7, 1.3), contrast=(0.85, 1.3), saturation=(0.7, 1.3), hue=(-0.0278, 0.0278)))
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False)
)

In [53]:
full_web_top_30_loader = DataLoader(
    full_web_top_30_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,             # shuffle ensures different order each epoch
    num_workers=4,            # safer on macOS, avoids multiprocessing issues
    pin_memory=True,      
    generator=generator       # ensures reproducible shuffling
)

In [54]:
norm_sqrt_inv_weights_train = compute_class_weights(train_dataset, method="sqrt_inverse", normalize=True)
norm_sqrt_inv_weights_train_top_30 = compute_class_weights(web_top_30_dataset, method="sqrt_inverse", normalize=True)   
norm_sqrt_inv_weights_train_top_30[-1] = 0 # handel other class
norm_sqrt_inv_weights = (norm_sqrt_inv_weights_train + (2*norm_sqrt_inv_weights_train_top_30))/2

In [55]:
top_30_val_results161_180 = pd.read_csv("Training outputs/top_val_results/mobilenetv2_stageF_top_30_val_results161_180.csv")

# Step 1: get the subset dataframe
df = top_30_val_results161_180[
    (top_30_val_results161_180['Species'] != "Average Performance (Macro)") &
    (top_30_val_results161_180['Epoch'] == 180)
].copy()

# Step 2: build a mapping from species to recall multiplier
def get_multiplier(recall):
    if recall < 0.6:
        return 10
    elif recall < 0.7:
        return 8
    elif recall < 0.8:
        return 6
    elif recall < 0.9:
        return 4
    elif recall < 1:
        return 2
    else:
        return 1

df['Multiplier'] = df['Recall'].apply(get_multiplier)

# Step 3: make sure class order matches your dataset order
species_to_multiplier = dict(zip(df['Species'], df['Multiplier']))

# Step 4: apply the multipliers to your weight vector
adjusted_weights = norm_sqrt_inv_weights.clone() if hasattr(norm_sqrt_inv_weights, 'clone') else np.copy(norm_sqrt_inv_weights)

for idx, class_name in enumerate(train_dataset.classes):
    if class_name in species_to_multiplier:
        adjusted_weights[idx] *= species_to_multiplier[class_name]

In [56]:
df

,Species,Precision,Recall,F1-score,Epoch,Multiplier
646,Anabasis articulata (Forssk.) Moq.,0.976000,0.983871,0.979920,180,2
647,Anabasis setifera Moq.,0.867403,0.792929,0.828496,180,6
648,Atriplex halimus L.,0.732938,0.879004,0.799353,180,4
649,Calotropis procera,0.961538,0.925926,0.943396,180,2
650,Capparis spinosa L.,0.947761,0.894366,0.920290,180,4
651,Cebatha pendula (J.R.Forst. & G.Forst.) Kuntze,0.857143,0.950000,0.901186,180,2
652,Cenchrus divisus,1.000000,0.977011,0.988372,180,2
653,Deverra tortuosa (Desf.) DC.,0.827160,0.957143,0.887417,180,2
654,Deverra triradiata Hochst. ex Boiss.,0.746269,1.000000,0.854701,180,1
655,Diplotaxis harra (Forssk.) Boiss.,0.903704,0.897059,0.900369,180,4


In [60]:
adjusted_weights, norm_sqrt_inv_weights

(tensor([ 2.1876,  6.8274,  6.0075,  2.4275,  4.5935,  2.5176,  3.1295,  3.7970,
          2.4456,  4.1975,  3.2993,  8.3270,  6.3478,  3.9639,  2.0345,  2.8870,
          2.7015,  9.3570,  6.7176,  5.5509,  1.9259,  1.5881,  1.1269,  0.8456,
          1.2772,  2.1456, 11.5294, 11.0180,  0.8796,  2.4793,  3.4599,  5.1923,
          1.9130], device='mps:0'),
 tensor([1.0938, 1.1379, 1.5019, 1.2137, 1.1484, 1.2588, 1.5647, 1.8985, 2.4456,
         1.0494, 1.6496, 2.0818, 1.0580, 0.9910, 1.0173, 1.4435, 2.7015, 1.5595,
         1.1196, 0.9252, 0.9630, 1.5881, 1.1269, 0.8456, 1.2772, 1.0728, 1.4412,
         1.1018, 0.8796, 1.2396, 1.7300, 2.5961, 0.4783], device='mps:0'))

In [61]:
norm_sqrt_inv_criterion = nn.CrossEntropyLoss(weight=adjusted_weights)
stageF_optimizer = torch.optim.AdamW(mobilenetv2_stageF_top_30.parameters(), lr=1e-5,
                            weight_decay=1e-4, betas=(0.9, 0.999))

In [62]:
start_epoch = 181
end_epoch = 195

In [63]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_top_30, full_web_top_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_top_30', save_dir = 'Training outputs',
                         class_weights='adjusted_norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 181 - Training Started...

Epoch 181, Batch 100, Loss: 0.0874, Accuracy: 96.47%, Time Passed: 3.66m
Epoch 181, Batch 200, Loss: 0.0903, Accuracy: 96.20%, Time Passed: 7.33m
Epoch 181, Batch 300, Loss: 0.0870, Accuracy: 96.27%, Time Passed: 11.18m
Epoch 181, Batch 400, Loss: 0.0897, Accuracy: 96.12%, Time Passed: 15.20m
Epoch 181, Batch 500, Loss: 0.0903, Accuracy: 96.01%, Time Passed: 19.13m
Epoch 181 Completed - Average Loss: 0.0891, Accuracy: 96.04%, Epoch Time: 21.93m
Training Complete Epoch 181 - Total Time: 21.93m

✅ Epoch 181 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.4968, Accuracy: 87.03%, Time Passed: 4.00m
Evaluation Complete - Loss: 0.4756, Accuracy: 87.97%, Total Time: 7.25m

📊 Epoch 181 - Evaluation Completed.


🚀 Epoch 182 - Training Started...

Epoch 182, Batch 100, Loss: 0.0932, Accuracy: 95.81%, Time Passed: 4.09m
Epoch 182, Batch 200, Loss: 0.0861, Accuracy: 96.14%, Time Passed: 7.96m
Epoch 182, Batch 300, Loss: 0.0856, Accuracy: 96.22%,

In [64]:
if training_states:
    joblib.dump(training_states, f'Training outputs/training_states_top_30_{start_epoch}_{end_epoch}.pkl')

In [65]:
del training_states

In [66]:
start_epoch = 196
end_epoch = 210

In [67]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_top_30, full_web_top_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_top_30', save_dir = 'Training outputs',
                         class_weights='adjusted_norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 196 - Training Started...

Epoch 196, Batch 100, Loss: 0.0605, Accuracy: 97.31%, Time Passed: 3.41m
Epoch 196, Batch 200, Loss: 0.0620, Accuracy: 97.20%, Time Passed: 6.85m
Epoch 196, Batch 300, Loss: 0.0577, Accuracy: 97.43%, Time Passed: 10.69m
Epoch 196, Batch 400, Loss: 0.0589, Accuracy: 97.33%, Time Passed: 14.26m
Epoch 196, Batch 500, Loss: 0.0598, Accuracy: 97.22%, Time Passed: 17.79m
Epoch 196 Completed - Average Loss: 0.0580, Accuracy: 97.29%, Epoch Time: 20.45m
Training Complete Epoch 196 - Total Time: 20.45m

✅ Epoch 196 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.4599, Accuracy: 88.12%, Time Passed: 3.92m
Evaluation Complete - Loss: 0.4403, Accuracy: 88.78%, Total Time: 7.00m

📊 Epoch 196 - Evaluation Completed.


🚀 Epoch 197 - Training Started...

Epoch 197, Batch 100, Loss: 0.0527, Accuracy: 97.41%, Time Passed: 3.57m
Epoch 197, Batch 200, Loss: 0.0578, Accuracy: 97.27%, Time Passed: 7.26m
Epoch 197, Batch 300, Loss: 0.0578, Accuracy: 97.33%,

In [69]:
if training_states:
    joblib.dump(training_states, f'Training outputs/training_states_top_30_{start_epoch}_{end_epoch}.pkl')

In [70]:
del training_states

In [71]:
start_epoch = 211
end_epoch = 220

In [72]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_top_30, full_web_top_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_top_30', save_dir = 'Training outputs',
                         class_weights='adjusted_norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 211 - Training Started...

Epoch 211, Batch 100, Loss: 0.0448, Accuracy: 97.31%, Time Passed: 4.09m
Epoch 211, Batch 200, Loss: 0.0439, Accuracy: 97.39%, Time Passed: 8.78m
Epoch 211, Batch 300, Loss: 0.0455, Accuracy: 97.49%, Time Passed: 13.21m
Epoch 211, Batch 400, Loss: 0.0462, Accuracy: 97.51%, Time Passed: 17.90m
Epoch 211, Batch 500, Loss: 0.0469, Accuracy: 97.53%, Time Passed: 22.54m
Epoch 211 Completed - Average Loss: 0.0461, Accuracy: 97.55%, Epoch Time: 25.96m
Training Complete Epoch 211 - Total Time: 25.96m

✅ Epoch 211 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.4568, Accuracy: 87.94%, Time Passed: 4.28m
Evaluation Complete - Loss: 0.4410, Accuracy: 88.80%, Total Time: 7.78m

📊 Epoch 211 - Evaluation Completed.


🚀 Epoch 212 - Training Started...

Epoch 212, Batch 100, Loss: 0.0412, Accuracy: 97.94%, Time Passed: 6.89m
Epoch 212, Batch 200, Loss: 0.0478, Accuracy: 97.77%, Time Passed: 13.73m
Epoch 212, Batch 300, Loss: 0.0493, Accuracy: 97.75%

python(43374) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43376) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43378) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43380) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Batch 100, Loss: 0.4639, Accuracy: 87.41%, Time Passed: 4.22m
Evaluation Complete - Loss: 0.4282, Accuracy: 88.91%, Total Time: 7.48m

📊 Epoch 212 - Evaluation Completed.


🚀 Epoch 213 - Training Started...



python(43439) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43441) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43443) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43445) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Epoch 213, Batch 100, Loss: 0.0396, Accuracy: 98.22%, Time Passed: 5.25m
Epoch 213, Batch 200, Loss: 0.0489, Accuracy: 97.86%, Time Passed: 10.70m
Epoch 213, Batch 300, Loss: 0.0499, Accuracy: 97.80%, Time Passed: 15.96m
Epoch 213, Batch 400, Loss: 0.0487, Accuracy: 97.75%, Time Passed: 20.59m
Epoch 213, Batch 500, Loss: 0.0476, Accuracy: 97.78%, Time Passed: 25.08m
Epoch 213 Completed - Average Loss: 0.0475, Accuracy: 97.78%, Epoch Time: 28.14m
Training Complete Epoch 213 - Total Time: 28.14m

✅ Epoch 213 - Training Completed. Starting Evaluation...



...

Batch 100, Loss: 0.4472, Accuracy: 88.44%, Time Passed: 4.13m
Evaluation Complete - Loss: 0.4408, Accuracy: 89.01%, Total Time: 7.43m

📊 Epoch 219 - Evaluation Completed.


🚀 Epoch 220 - Training Started...



python(47699) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(47701) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(47703) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(47705) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Epoch 220, Batch 100, Loss: 0.0355, Accuracy: 98.28%, Time Passed: 5.88m
Epoch 220, Batch 200, Loss: 0.0382, Accuracy: 98.22%, Time Passed: 10.88m
Epoch 220, Batch 300, Loss: 0.0413, Accuracy: 98.10%, Time Passed: 15.80m
Epoch 220, Batch 400, Loss: 0.0418, Accuracy: 98.00%, Time Passed: 20.46m
Epoch 220, Batch 500, Loss: 0.0426, Accuracy: 97.96%, Time Passed: 24.83m
Epoch 220 Completed - Average Loss: 0.0431, Accuracy: 97.93%, Epoch Time: 28.47m
Training Complete Epoch 220 - Total Time: 28.47m

✅ Epoch 220 - Training Completed. Starting Evaluation...



python(47967) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(47970) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(47973) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(47976) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Batch 100, Loss: 0.4464, Accuracy: 88.44%, Time Passed: 4.13m
Evaluation Complete - Loss: 0.4387, Accuracy: 88.95%, Total Time: 7.38m

📊 Epoch 220 - Evaluation Completed.



In [73]:
if training_states:
    joblib.dump(training_states, f'Training outputs/training_states_top_30_{start_epoch}_{end_epoch}.pkl')

## **full_web_lowest_30_dataset**

### Step 1 

In [46]:
state, generator = setup_reproducibility(start_fresh=True, seed=seeds[1])

✅ Random seed set to 1234 (MPS available: True)


In [77]:
training_states = joblib.load(WORKSPACE_DIR /
                              "Phase 1/mobilenetv2/training_states/training_states_121_130.pkl")
 
mobilenetv2_stageF_lowest_30 = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes)) 
device = torch.device("mps") 
mobilenetv2_stageF_lowest_30.to(device)
mobilenetv2_stageF_lowest_30.load_state_dict(training_states[127]["model_state_dict"])
del training_states

### the following method divide the other class weights by 2

In [48]:
norm_sqrt_inv_weights_train = compute_class_weights(train_dataset, method="sqrt_inverse", normalize=True)
norm_sqrt_inv_weights_train_lowest_30 = compute_class_weights(web_lowest_30_dataset, method="sqrt_inverse", normalize=True)   
norm_sqrt_inv_weights_train_lowest_30[-1] = 0 # handel other class

In [49]:
norm_sqrt_inv_weights = (norm_sqrt_inv_weights_train + (2*norm_sqrt_inv_weights_train_lowest_30))/2

In [50]:
norm_sqrt_inv_weights

tensor([1.0964, 1.1421, 1.5081, 1.2170, 1.1522, 1.2750, 1.5694, 1.9039, 2.3485,
        1.0524, 1.6564, 2.0909, 1.0611, 0.9941, 1.0207, 1.4491, 2.7100, 1.5648,
        1.1239, 0.9281, 0.9572, 1.5909, 1.1301, 0.8475, 1.2812, 1.0702, 1.4472,
        1.1048, 0.8819, 1.2444, 1.6734, 2.6062, 0.4783], device='mps:0')

In [51]:
norm_sqrt_inv_criterion = nn.CrossEntropyLoss(weight=norm_sqrt_inv_weights)
stageF_optimizer = torch.optim.AdamW(mobilenetv2_stageF_lowest_30.parameters(), lr=1e-5,
                            weight_decay=1e-4, betas=(0.9, 0.999))

In [52]:
start_epoch = 128
end_epoch = 140
start_epoch, end_epoch

(128, 140)

In [53]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_lowest_30, full_web_lowest_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_lowest_30', save_dir = 'Training outputs',
                         class_weights='norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 128 - Training Started...

Epoch 128, Batch 100, Loss: 1.0329, Accuracy: 85.28%, Time Passed: 3.85m
Epoch 128, Batch 200, Loss: 0.9729, Accuracy: 85.67%, Time Passed: 7.84m
Epoch 128, Batch 300, Loss: 0.9540, Accuracy: 85.36%, Time Passed: 11.84m
Epoch 128, Batch 400, Loss: 0.9084, Accuracy: 85.72%, Time Passed: 16.27m
Epoch 128, Batch 500, Loss: 0.8916, Accuracy: 85.62%, Time Passed: 20.44m
Epoch 128 Completed - Average Loss: 0.8734, Accuracy: 85.70%, Epoch Time: 23.53m
Training Complete Epoch 128 - Total Time: 23.53m

✅ Epoch 128 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.7620, Accuracy: 79.69%, Time Passed: 4.26m
Evaluation Complete - Loss: 0.7217, Accuracy: 82.05%, Total Time: 7.63m

📊 Epoch 128 - Evaluation Completed.


🚀 Epoch 129 - Training Started...

Epoch 129, Batch 100, Loss: 0.6852, Accuracy: 85.97%, Time Passed: 4.31m
Epoch 129, Batch 200, Loss: 0.6766, Accuracy: 86.11%, Time Passed: 8.85m
Epoch 129, Batch 300, Loss: 0.6557, Accuracy: 86.31%,

In [54]:
if training_states:
    joblib.dump(training_states, f'Training outputs/training_states_lowest_30_{start_epoch}_{end_epoch}.pkl')

In [55]:
del training_states

In [56]:
state, generator = setup_reproducibility(continue_run=True, path="Training outputs/rng_state_140.pkl", mode="save", seed=seeds[1], generator=generator)

✅ RNG state saved to Training outputs/rng_state_140.pkl


In [57]:
start_epoch = 141
end_epoch = 150
start_epoch, end_epoch

(141, 150)

In [58]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_lowest_30, full_web_lowest_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_lowest_30', save_dir = 'Training outputs',
                         class_weights='norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 141 - Training Started...

Epoch 141, Batch 100, Loss: 0.3320, Accuracy: 90.41%, Time Passed: 3.80m
Epoch 141, Batch 200, Loss: 0.3550, Accuracy: 89.80%, Time Passed: 7.49m
Epoch 141, Batch 300, Loss: 0.3558, Accuracy: 89.57%, Time Passed: 11.29m
Epoch 141, Batch 400, Loss: 0.3487, Accuracy: 89.71%, Time Passed: 15.15m
Epoch 141, Batch 500, Loss: 0.3496, Accuracy: 89.69%, Time Passed: 19.13m
Epoch 141 Completed - Average Loss: 0.3496, Accuracy: 89.64%, Epoch Time: 22.01m
Training Complete Epoch 141 - Total Time: 22.01m

✅ Epoch 141 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.5481, Accuracy: 83.41%, Time Passed: 4.00m
Evaluation Complete - Loss: 0.5127, Accuracy: 85.53%, Total Time: 7.11m

📊 Epoch 141 - Evaluation Completed.


🚀 Epoch 142 - Training Started...

Epoch 142, Batch 100, Loss: 0.2985, Accuracy: 91.56%, Time Passed: 4.49m
Epoch 142, Batch 200, Loss: 0.3256, Accuracy: 90.88%, Time Passed: 8.60m
Epoch 142, Batch 300, Loss: 0.3315, Accuracy: 90.55%,

python(37440) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(37442) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(37444) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(37446) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Batch 100, Loss: 0.5724, Accuracy: 83.16%, Time Passed: 4.59m
Evaluation Complete - Loss: 0.5363, Accuracy: 85.44%, Total Time: 7.98m

📊 Epoch 144 - Evaluation Completed.


🚀 Epoch 145 - Training Started...



python(37475) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(37477) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(37479) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(37481) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Epoch 145, Batch 100, Loss: 0.3645, Accuracy: 89.69%, Time Passed: 4.94m
Epoch 145, Batch 200, Loss: 0.3480, Accuracy: 89.98%, Time Passed: 10.27m
Epoch 145, Batch 300, Loss: 0.3446, Accuracy: 90.09%, Time Passed: 15.82m
Epoch 145, Batch 400, Loss: 0.3353, Accuracy: 90.23%, Time Passed: 20.97m
Epoch 145, Batch 500, Loss: 0.3317, Accuracy: 90.32%, Time Passed: 26.07m
Epoch 145 Completed - Average Loss: 0.3263, Accuracy: 90.48%, Epoch Time: 29.75m
Training Complete Epoch 145 - Total Time: 29.75m

✅ Epoch 145 - Training Completed. Starting Evaluation...



...

Batch 100, Loss: 0.5872, Accuracy: 82.97%, Time Passed: 4.36m
Evaluation Complete - Loss: 0.5307, Accuracy: 85.27%, Total Time: 7.82m

📊 Epoch 149 - Evaluation Completed.


🚀 Epoch 150 - Training Started...



python(40094) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(40096) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(40098) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(40100) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Epoch 150, Batch 100, Loss: 0.2775, Accuracy: 91.53%, Time Passed: 5.39m
Epoch 150, Batch 200, Loss: 0.2830, Accuracy: 91.56%, Time Passed: 10.00m
Epoch 150, Batch 300, Loss: 0.2885, Accuracy: 91.50%, Time Passed: 14.79m
Epoch 150, Batch 400, Loss: 0.2866, Accuracy: 91.45%, Time Passed: 19.56m
Epoch 150, Batch 500, Loss: 0.2864, Accuracy: 91.47%, Time Passed: 24.33m
Epoch 150 Completed - Average Loss: 0.2887, Accuracy: 91.36%, Epoch Time: 27.68m
Training Complete Epoch 150 - Total Time: 27.68m

✅ Epoch 150 - Training Completed. Starting Evaluation...



python(40171) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(40173) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(40175) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(40177) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Batch 100, Loss: 0.5543, Accuracy: 83.25%, Time Passed: 4.35m
Evaluation Complete - Loss: 0.5240, Accuracy: 85.34%, Total Time: 7.76m

📊 Epoch 150 - Evaluation Completed.



In [59]:
if training_states:
    joblib.dump(training_states, f'Training outputs/training_states_lowest_30_{start_epoch}_{end_epoch}.pkl')

In [60]:
del training_states

In [51]:
training_states = joblib.load("Training outputs/training_states_lowest_30_141_150.pkl"
)
mobilenetv2_stageF_lowest_30 = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes)) 
device = torch.device("mps") 
mobilenetv2_stageF_lowest_30.to(device)
mobilenetv2_stageF_lowest_30.load_state_dict(training_states[150]["model_state_dict"])
del training_states

norm_sqrt_inv_weights_train = compute_class_weights(train_dataset, method="sqrt_inverse", normalize=True)
norm_sqrt_inv_weights_train_lowest_30 = compute_class_weights(web_lowest_30_dataset, method="sqrt_inverse", normalize=True)   
norm_sqrt_inv_weights_train_lowest_30[-1] = 0 # handel other class
norm_sqrt_inv_weights = (norm_sqrt_inv_weights_train + (2*norm_sqrt_inv_weights_train_lowest_30))/2
norm_sqrt_inv_criterion = nn.CrossEntropyLoss(weight=norm_sqrt_inv_weights)

web_datasets_dict_list = [{'All Web Data': web_dataset},
 {'Web Correctly Classified': web_correctly_classified_dataset},
 {'Web Top 30': web_top_30_dataset}, 
 {'Web Mid_30_70': web_mid_30_70_dataset},
 {'Web Lowest 30': web_lowest_30_dataset}
]

In [52]:
lowest_30_all_results, lowest_30_metrics_summary = evaluate_on_three_sets(
    mobilenetv2_stageF_lowest_30,
    web_datasets_dict_list[4],          
    web_datasets_dict_list[0],        
    [d for i, d in enumerate(web_datasets_dict_list) if i not in (0, 4)],      # list of dicts, each has one dataset
    val_transform,
    "mobilenetv2_stageF_lowest_30",
    norm_sqrt_inv_criterion,
    train_dataset.classes,
    epoch_num=150,
    log_interval=100,
    batch_size=32,
    num_workers=4
)


🔹 Starting evaluation for: Web Lowest 30 (used in training)
Evaluation Complete - Loss: 1.8633, Accuracy: 50.96%, Total Time: 0.55m

🔹 Starting evaluation for: All Web Data
Batch 100, Loss: 1.4624, Accuracy: 59.34%, Time Passed: 0.37m
Evaluation Complete - Loss: 1.3854, Accuracy: 60.73%, Total Time: 0.92m

🔹 Starting evaluation for: Web Correctly Classified
Evaluation Complete - Loss: 7.0380, Accuracy: 28.32%, Total Time: 0.59m

🔹 Starting evaluation for: Web Top 30
Evaluation Complete - Loss: 1.3445, Accuracy: 58.42%, Total Time: 0.54m

🔹 Starting evaluation for: Web Mid_30_70
Evaluation Complete - Loss: 1.9630, Accuracy: 46.81%, Total Time: 0.57m

🔹 Starting evaluation for: Concatenated Remaining Web Data (unseen)
Batch 100, Loss: 4.5967, Accuracy: 41.31%, Time Passed: 0.43m
Evaluation Complete - Loss: 3.7918, Accuracy: 42.56%, Total Time: 0.89m


In [143]:
lowest_30_all_results.to_csv("Training outputs/lowest_30_web_test_150_results.csv", index=False)
lowest_30_metrics_summary.to_csv("Training outputs/lowest_30_web_test_150_metrics.csv", index=False)

### Continue training (Step 2)

In [53]:
state, generator = setup_reproducibility(start_fresh=True, seed=seeds[2])

✅ Random seed set to 2025 (MPS available: True)


In [54]:
training_states = joblib.load("Training outputs/training_states_lowest_30_141_150.pkl")
mobilenetv2_stageF_lowest_30 = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes)) 
device = torch.device("mps") 
mobilenetv2_stageF_lowest_30.to(device)
mobilenetv2_stageF_lowest_30.load_state_dict(training_states[150]["model_state_dict"])
del training_states

In [55]:
norm_sqrt_inv_weights_train = compute_class_weights(train_dataset, method="sqrt_inverse", normalize=True)
norm_sqrt_inv_weights_train_lowest_30 = compute_class_weights(web_lowest_30_dataset, method="sqrt_inverse", normalize=True)   
norm_sqrt_inv_weights_train_lowest_30[-1] = 0 # handel other class
norm_sqrt_inv_weights = (norm_sqrt_inv_weights_train + (2*norm_sqrt_inv_weights_train_lowest_30))/2


lowest_30_val_results141_150 = pd.read_csv("Training outputs/mobilenetv2_stageF_lowest_30_val_results141_150.csv")

# Step 1: get the subset dataframe
df = lowest_30_val_results141_150[
    (lowest_30_val_results141_150['Species'] != "Average Performance (Macro)") &
    (lowest_30_val_results141_150['Epoch'] == 150)
].copy()

# Step 2: build a mapping from species to recall multiplier
def get_multiplier(recall):
    if recall < 0.6:
        return 10
    elif recall < 0.7:
        return 8
    elif recall < 0.8:
        return 6
    elif recall < 0.9:
        return 4
    elif recall < 1:
        return 2
    else:
        return 1

df['Multiplier'] = df['Recall'].apply(get_multiplier)

# Step 3: make sure class order matches your dataset order
species_to_multiplier = dict(zip(df['Species'], df['Multiplier']))

# Step 4: apply the multipliers to your weight vector
adjusted_weights = norm_sqrt_inv_weights.clone() if hasattr(norm_sqrt_inv_weights, 'clone') else np.copy(norm_sqrt_inv_weights)

for idx, class_name in enumerate(train_dataset.classes):
    if class_name in species_to_multiplier:
        adjusted_weights[idx] *= species_to_multiplier[class_name]

In [56]:
df

,Species,Precision,Recall,F1-score,Epoch,Multiplier
306,Anabasis articulata (Forssk.) Moq.,0.849315,1.000000,0.918519,150,1
307,Anabasis setifera Moq.,0.900000,0.727273,0.804469,150,6
308,Atriplex halimus L.,0.718023,0.879004,0.790400,150,4
309,Calotropis procera,0.916667,0.950617,0.933333,150,2
310,Capparis spinosa L.,0.962617,0.725352,0.827309,150,6
311,Cebatha pendula (J.R.Forst. & G.Forst.) Kuntze,0.932584,0.691667,0.794258,150,8
312,Cenchrus divisus,0.943182,0.954023,0.948571,150,2
313,Deverra tortuosa (Desf.) DC.,0.813953,1.000000,0.897436,150,1
314,Deverra triradiata Hochst. ex Boiss.,0.793651,1.000000,0.884956,150,1
315,Diplotaxis harra (Forssk.) Boiss.,0.827586,0.882353,0.854093,150,4


In [57]:
adjusted_weights, norm_sqrt_inv_weights

(tensor([ 1.0964,  6.8528,  6.0324,  2.4340,  6.9130, 10.2000,  3.1388,  1.9039,
          2.3485,  4.2096,  3.3128,  8.3636, 10.6105,  1.9881,  4.0826,  5.7964,
          5.4200,  9.3887,  6.7435,  5.5686,  1.9145,  1.5909,  1.1301,  1.6951,
          1.2812,  2.1405, 11.5773, 11.0477,  0.8819,  2.4888,  6.6937,  5.2123,
          2.8695], device='mps:0'),
 tensor([1.0964, 1.1421, 1.5081, 1.2170, 1.1522, 1.2750, 1.5694, 1.9039, 2.3485,
         1.0524, 1.6564, 2.0909, 1.0611, 0.9941, 1.0207, 1.4491, 2.7100, 1.5648,
         1.1239, 0.9281, 0.9572, 1.5909, 1.1301, 0.8475, 1.2812, 1.0702, 1.4472,
         1.1048, 0.8819, 1.2444, 1.6734, 2.6062, 0.4783], device='mps:0'))

In [58]:
norm_sqrt_inv_criterion = nn.CrossEntropyLoss(weight=adjusted_weights)
stageF_optimizer = torch.optim.AdamW(mobilenetv2_stageF_lowest_30.parameters(), lr=1e-5,
                            weight_decay=1e-4, betas=(0.9, 0.999))

In [59]:
start_epoch = 151
end_epoch = 160
start_epoch, end_epoch

(151, 160)

In [60]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_lowest_30, full_web_lowest_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_lowest_30', save_dir = 'Training outputs',
                         class_weights='adjusted_norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 151 - Training Started...

Epoch 151, Batch 100, Loss: 0.2772, Accuracy: 91.22%, Time Passed: 3.41m
Epoch 151, Batch 200, Loss: 0.2817, Accuracy: 91.08%, Time Passed: 6.91m
Epoch 151, Batch 300, Loss: 0.2726, Accuracy: 91.26%, Time Passed: 10.65m
Epoch 151, Batch 400, Loss: 0.2729, Accuracy: 91.09%, Time Passed: 14.50m
Epoch 151, Batch 500, Loss: 0.2719, Accuracy: 91.01%, Time Passed: 18.43m
Epoch 151 Completed - Average Loss: 0.2730, Accuracy: 90.94%, Epoch Time: 21.13m
Training Complete Epoch 151 - Total Time: 21.13m

✅ Epoch 151 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.5077, Accuracy: 85.12%, Time Passed: 4.32m
Evaluation Complete - Loss: 0.4836, Accuracy: 86.81%, Total Time: 7.53m

📊 Epoch 151 - Evaluation Completed.


🚀 Epoch 152 - Training Started...

Epoch 152, Batch 100, Loss: 0.2157, Accuracy: 92.25%, Time Passed: 3.65m
Epoch 152, Batch 200, Loss: 0.2260, Accuracy: 91.73%, Time Passed: 7.42m
Epoch 152, Batch 300, Loss: 0.2389, Accuracy: 91.46%,

In [62]:
if training_states:
    joblib.dump(training_states, f'Training outputs/training_states_lowest_30_{start_epoch}_{end_epoch}.pkl')

In [63]:
del training_states

### Continue training

In [57]:
state, generator = setup_reproducibility(start_fresh=True, seed=seeds[0])

✅ Random seed set to 42 (MPS available: True)


In [58]:
training_states = joblib.load("Training outputs/training_states_lowest_30_151_160.pkl")
mobilenetv2_stageF_lowest_30 = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes)) 
device = torch.device("mps") 
mobilenetv2_stageF_lowest_30.to(device)
mobilenetv2_stageF_lowest_30.load_state_dict(training_states[160]["model_state_dict"])
del training_states

In [59]:
norm_sqrt_inv_weights_train = compute_class_weights(train_dataset, method="sqrt_inverse", normalize=True)
norm_sqrt_inv_weights_train_lowest_30 = compute_class_weights(web_lowest_30_dataset, method="sqrt_inverse", normalize=True)   
norm_sqrt_inv_weights_train_lowest_30[-1] = 0 # handel other class
norm_sqrt_inv_weights = (norm_sqrt_inv_weights_train + (2*norm_sqrt_inv_weights_train_lowest_30))/2


lowest_30_val_results151_160 = pd.read_csv("Training outputs/mobilenetv2_stageF_lowest_30_val_results151_160.csv")

# Step 1: get the subset dataframe
df = lowest_30_val_results151_160[
    (lowest_30_val_results151_160['Species'] != "Average Performance (Macro)") &
    (lowest_30_val_results151_160['Epoch'] == 160)
].copy()

# Step 2: build a mapping from species to recall multiplier
def get_multiplier(recall):
    if recall < 0.6:
        return 10
    elif recall < 0.7:
        return 8
    elif recall < 0.8:
        return 6
    elif recall < 0.9:
        return 4
    elif recall < 1:
        return 2
    else:
        return 1

df['Multiplier'] = df['Recall'].apply(get_multiplier)

# Step 3: make sure class order matches your dataset order
species_to_multiplier = dict(zip(df['Species'], df['Multiplier']))

# Step 4: apply the multipliers to your weight vector
adjusted_weights = norm_sqrt_inv_weights.clone() if hasattr(norm_sqrt_inv_weights, 'clone') else np.copy(norm_sqrt_inv_weights)

for idx, class_name in enumerate(train_dataset.classes):
    if class_name in species_to_multiplier:
        adjusted_weights[idx] *= species_to_multiplier[class_name]

In [60]:
df

,Species,Precision,Recall,F1-score,Epoch,Multiplier
306,Anabasis articulata (Forssk.) Moq.,0.930769,0.975806,0.952756,160,2
307,Anabasis setifera Moq.,0.857923,0.792929,0.824147,160,6
308,Atriplex halimus L.,0.752294,0.875445,0.809211,160,4
309,Calotropis procera,0.962025,0.938272,0.950000,160,2
310,Capparis spinosa L.,0.958333,0.809859,0.877863,160,4
311,Cebatha pendula (J.R.Forst. & G.Forst.) Kuntze,0.856000,0.891667,0.873469,160,4
312,Cenchrus divisus,1.000000,0.977011,0.988372,160,2
313,Deverra tortuosa (Desf.) DC.,0.909091,1.000000,0.952381,160,1
314,Deverra triradiata Hochst. ex Boiss.,0.980392,1.000000,0.990099,160,1
315,Diplotaxis harra (Forssk.) Boiss.,0.846154,0.889706,0.867384,160,4


In [61]:
adjusted_weights, norm_sqrt_inv_weights

(tensor([ 2.1927,  6.8528,  6.0324,  2.4340,  4.6086,  5.1000,  3.1388,  1.9039,
          2.3485,  4.2096,  3.3128,  8.3636,  8.4884,  1.9881,  2.0413,  2.8982,
          2.7100, 12.5182,  6.7435,  3.7124,  1.9145,  1.5909,  1.1301,  1.6951,
          2.5624,  2.1405, 11.5773, 11.0477,  0.8819,  2.4888,  3.3469,  2.6062,
          1.9130], device='mps:0'),
 tensor([1.0964, 1.1421, 1.5081, 1.2170, 1.1522, 1.2750, 1.5694, 1.9039, 2.3485,
         1.0524, 1.6564, 2.0909, 1.0611, 0.9941, 1.0207, 1.4491, 2.7100, 1.5648,
         1.1239, 0.9281, 0.9572, 1.5909, 1.1301, 0.8475, 1.2812, 1.0702, 1.4472,
         1.1048, 0.8819, 1.2444, 1.6734, 2.6062, 0.4783], device='mps:0'))

In [62]:
norm_sqrt_inv_criterion = nn.CrossEntropyLoss(weight=adjusted_weights)
stageF_optimizer = torch.optim.AdamW(mobilenetv2_stageF_lowest_30.parameters(), lr=1e-5,
                            weight_decay=1e-4, betas=(0.9, 0.999))

In [63]:
start_epoch = 161
end_epoch = 180
start_epoch, end_epoch

(161, 180)

In [64]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_lowest_30, full_web_lowest_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_lowest_30', save_dir = 'Training outputs',
                         class_weights='adjusted_norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 161 - Training Started...

Epoch 161, Batch 100, Loss: 0.1907, Accuracy: 92.84%, Time Passed: 3.59m
Epoch 161, Batch 200, Loss: 0.2004, Accuracy: 92.45%, Time Passed: 7.22m
Epoch 161, Batch 300, Loss: 0.1940, Accuracy: 92.33%, Time Passed: 10.97m
Epoch 161, Batch 400, Loss: 0.1966, Accuracy: 92.35%, Time Passed: 14.62m
Epoch 161, Batch 500, Loss: 0.1955, Accuracy: 92.38%, Time Passed: 18.40m
Epoch 161 Completed - Average Loss: 0.1972, Accuracy: 92.21%, Epoch Time: 21.24m
Training Complete Epoch 161 - Total Time: 21.24m

✅ Epoch 161 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.4929, Accuracy: 86.25%, Time Passed: 4.10m
Evaluation Complete - Loss: 0.4661, Accuracy: 87.52%, Total Time: 7.37m

📊 Epoch 161 - Evaluation Completed.


🚀 Epoch 162 - Training Started...

Epoch 162, Batch 100, Loss: 0.1557, Accuracy: 93.38%, Time Passed: 3.73m
Epoch 162, Batch 200, Loss: 0.1660, Accuracy: 92.88%, Time Passed: 7.70m
Epoch 162, Batch 300, Loss: 0.1764, Accuracy: 92.80%,

In [66]:
if training_states:
    joblib.dump(training_states, f'Training outputs/training_states_lowest_30_{start_epoch}_{end_epoch}.pkl')

In [67]:
del training_states

### New Method 

In [35]:
state, generator = setup_reproducibility(start_fresh=True, seed=seeds[2])

✅ Random seed set to 2025 (MPS available: True)


In [36]:
training_states = joblib.load("Training outputs/training_states/training_states_lowest_30_161_180.pkl")
mobilenetv2_stageF_lowest_30 = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes)) 
device = torch.device("mps") 
mobilenetv2_stageF_lowest_30.to(device)
mobilenetv2_stageF_lowest_30.load_state_dict(training_states[180]["model_state_dict"])
del training_states

In [53]:
default_train_transform = build_train_transform(crop_type="random", scale=(0.2, 1.0))
train_dataset.transform = default_train_transform  
web_lowest_30_dataset.transform = default_train_transform
val_dataset.transform = val_transform  
full_web_lowest_30_dataset = ConcatDataset([train_dataset, web_lowest_30_dataset])
train_dataset.transform

Compose(
      ToImage()
      RandomResizedCrop(size=(224, 224), scale=(0.2, 1.0), ratio=(0.75, 1.33), interpolation=InterpolationMode.BILINEAR, antialias=True)
      RandomHorizontalFlip(p=0.5)
      RandomVerticalFlip(p=0.2)
      RandomApply(    RandomRotation(degrees=[-90.0, 90.0], interpolation=InterpolationMode.BILINEAR, expand=False, fill=0))
      ToDtype(scale=True)
      RandomApply(    ColorJitter(brightness=(0.7, 1.3), contrast=(0.85, 1.3), saturation=(0.7, 1.3), hue=(-0.0278, 0.0278)))
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False)
)

In [54]:
full_web_lowest_30_loader = DataLoader(
    full_web_lowest_30_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,             # shuffle ensures different order each epoch
    num_workers=4,            # safer on macOS, avoids multiprocessing issues
    pin_memory=True,      
    generator=generator       # ensures reproducible shuffling
)

In [55]:
norm_sqrt_inv_weights_train = compute_class_weights(train_dataset, method="sqrt_inverse", normalize=True)
norm_sqrt_inv_weights_train_lowest_30 = compute_class_weights(web_lowest_30_dataset, method="sqrt_inverse", normalize=True)   
norm_sqrt_inv_weights_train_lowest_30[-1] = 0 # handel other class
norm_sqrt_inv_weights = (norm_sqrt_inv_weights_train + (2*norm_sqrt_inv_weights_train_lowest_30))/2


lowest_30_val_results161_180 = pd.read_csv("Training outputs/lowest_val_results/mobilenetv2_stageF_lowest_30_val_results161_180.csv")

# Step 1: get the subset dataframe
df = lowest_30_val_results161_180[
    (lowest_30_val_results161_180['Species'] != "Average Performance (Macro)") &
    (lowest_30_val_results161_180['Epoch'] == 180)
].copy()

# Step 2: build a mapping from species to recall multiplier
def get_multiplier(recall):
    if recall < 0.6:
        return 10
    elif recall < 0.7:
        return 8
    elif recall < 0.8:
        return 6
    elif recall < 0.9:
        return 4
    elif recall < 1:
        return 2
    else:
        return 1

df['Multiplier'] = df['Recall'].apply(get_multiplier)

# Step 3: make sure class order matches your dataset order
species_to_multiplier = dict(zip(df['Species'], df['Multiplier']))

# Step 4: apply the multipliers to your weight vector
adjusted_weights = norm_sqrt_inv_weights.clone() if hasattr(norm_sqrt_inv_weights, 'clone') else np.copy(norm_sqrt_inv_weights)

for idx, class_name in enumerate(train_dataset.classes):
    if class_name in species_to_multiplier:
        adjusted_weights[idx] *= species_to_multiplier[class_name]

In [56]:
df

,Species,Precision,Recall,F1-score,Epoch,Multiplier
646,Anabasis articulata (Forssk.) Moq.,0.897810,0.991935,0.942529,180,2
647,Anabasis setifera Moq.,0.871345,0.752525,0.807588,180,6
648,Atriplex halimus L.,0.688347,0.903915,0.781538,180,2
649,Calotropis procera,0.938272,0.938272,0.938272,180,2
650,Capparis spinosa L.,0.973214,0.767606,0.858268,180,6
651,Cebatha pendula (J.R.Forst. & G.Forst.) Kuntze,0.857143,0.850000,0.853556,180,4
652,Cenchrus divisus,1.000000,0.977011,0.988372,180,2
653,Deverra tortuosa (Desf.) DC.,0.875000,1.000000,0.933333,180,1
654,Deverra triradiata Hochst. ex Boiss.,0.909091,1.000000,0.952381,180,1
655,Diplotaxis harra (Forssk.) Boiss.,0.909091,0.882353,0.895522,180,4


In [57]:
adjusted_weights, norm_sqrt_inv_weights

(tensor([ 2.1927,  6.8528,  3.0162,  2.4340,  6.9130,  5.1000,  3.1388,  1.9039,
          2.3485,  4.2096,  3.3128,  8.3636,  8.4884,  1.9881,  2.0413,  2.8982,
          5.4200,  9.3887,  6.7435,  5.5686,  1.9145,  1.5909,  1.1301,  0.8475,
          1.2812,  2.1405, 11.5773,  8.8382,  0.8819,  2.4888,  3.3469,  5.2123,
          1.9130], device='mps:0'),
 tensor([1.0964, 1.1421, 1.5081, 1.2170, 1.1522, 1.2750, 1.5694, 1.9039, 2.3485,
         1.0524, 1.6564, 2.0909, 1.0611, 0.9941, 1.0207, 1.4491, 2.7100, 1.5648,
         1.1239, 0.9281, 0.9572, 1.5909, 1.1301, 0.8475, 1.2812, 1.0702, 1.4472,
         1.1048, 0.8819, 1.2444, 1.6734, 2.6062, 0.4783], device='mps:0'))

In [58]:
norm_sqrt_inv_criterion = nn.CrossEntropyLoss(weight=adjusted_weights)
stageF_optimizer = torch.optim.AdamW(mobilenetv2_stageF_lowest_30.parameters(), lr=1e-5,
                            weight_decay=1e-4, betas=(0.9, 0.999))

In [59]:
start_epoch = 181
end_epoch = 182
start_epoch, end_epoch

(181, 182)

In [60]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_lowest_30, full_web_lowest_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_lowest_30', save_dir = 'Training outputs',
                         class_weights='adjusted_norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 181 - Training Started...

Epoch 181, Batch 100, Loss: 0.1353, Accuracy: 94.44%, Time Passed: 3.75m
Epoch 181, Batch 200, Loss: 0.1371, Accuracy: 94.59%, Time Passed: 7.71m
Epoch 181, Batch 300, Loss: 0.1291, Accuracy: 94.69%, Time Passed: 11.83m
Epoch 181, Batch 400, Loss: 0.1295, Accuracy: 94.79%, Time Passed: 16.54m
Epoch 181, Batch 500, Loss: 0.1271, Accuracy: 94.92%, Time Passed: 21.59m
Epoch 181 Completed - Average Loss: 0.1260, Accuracy: 94.91%, Epoch Time: 25.27m
Training Complete Epoch 181 - Total Time: 25.27m

✅ Epoch 181 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.4938, Accuracy: 86.16%, Time Passed: 5.32m
Evaluation Complete - Loss: 0.4556, Accuracy: 87.52%, Total Time: 9.29m

📊 Epoch 181 - Evaluation Completed.


🚀 Epoch 182 - Training Started...

Epoch 182, Batch 100, Loss: 0.1299, Accuracy: 94.69%, Time Passed: 5.18m
Epoch 182, Batch 200, Loss: 0.1129, Accuracy: 95.14%, Time Passed: 10.85m
Epoch 182, Batch 300, Loss: 0.1116, Accuracy: 95.11%

In [61]:
if training_states:
    joblib.dump(training_states, f'Training outputs/training_states_lowest_30_{start_epoch}_{end_epoch}.pkl')

In [62]:
del training_states

In [64]:
start_epoch = 183
end_epoch = 190
start_epoch, end_epoch

(183, 190)

In [65]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_lowest_30, full_web_lowest_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_lowest_30', save_dir = 'Training outputs',
                         class_weights='adjusted_norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 183 - Training Started...

Epoch 183, Batch 100, Loss: 0.1011, Accuracy: 95.84%, Time Passed: 3.64m
Epoch 183, Batch 200, Loss: 0.1064, Accuracy: 95.48%, Time Passed: 7.59m
Epoch 183, Batch 300, Loss: 0.1080, Accuracy: 95.52%, Time Passed: 11.64m
Epoch 183, Batch 400, Loss: 0.1120, Accuracy: 95.34%, Time Passed: 16.18m
Epoch 183, Batch 500, Loss: 0.1055, Accuracy: 95.58%, Time Passed: 20.68m
Epoch 183 Completed - Average Loss: 0.1032, Accuracy: 95.60%, Epoch Time: 23.67m
Training Complete Epoch 183 - Total Time: 23.67m

✅ Epoch 183 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.4816, Accuracy: 86.19%, Time Passed: 4.26m
Evaluation Complete - Loss: 0.4273, Accuracy: 88.02%, Total Time: 7.74m

📊 Epoch 183 - Evaluation Completed.


🚀 Epoch 184 - Training Started...

Epoch 184, Batch 100, Loss: 0.0992, Accuracy: 95.78%, Time Passed: 4.03m
Epoch 184, Batch 200, Loss: 0.1012, Accuracy: 95.39%, Time Passed: 8.30m
Epoch 184, Batch 300, Loss: 0.0988, Accuracy: 95.47%,

In [66]:
if training_states:
    joblib.dump(training_states, f'Training outputs/training_states_lowest_30_{start_epoch}_{end_epoch}.pkl')

In [67]:
del training_states

In [69]:
start_epoch = 191
end_epoch = 200
start_epoch, end_epoch

(191, 200)

In [70]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_lowest_30, full_web_lowest_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_lowest_30', save_dir = 'Training outputs',
                         class_weights='adjusted_norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 191 - Training Started...

Epoch 191, Batch 100, Loss: 0.0930, Accuracy: 95.31%, Time Passed: 3.97m
Epoch 191, Batch 200, Loss: 0.0937, Accuracy: 95.55%, Time Passed: 7.88m
Epoch 191, Batch 300, Loss: 0.0908, Accuracy: 95.76%, Time Passed: 11.80m
Epoch 191, Batch 400, Loss: 0.0921, Accuracy: 95.78%, Time Passed: 15.71m
Epoch 191, Batch 500, Loss: 0.0883, Accuracy: 95.86%, Time Passed: 19.82m
Epoch 191 Completed - Average Loss: 0.0894, Accuracy: 95.82%, Epoch Time: 22.63m
Training Complete Epoch 191 - Total Time: 22.63m

✅ Epoch 191 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.4650, Accuracy: 86.78%, Time Passed: 4.14m
Evaluation Complete - Loss: 0.4287, Accuracy: 88.19%, Total Time: 7.61m

📊 Epoch 191 - Evaluation Completed.


🚀 Epoch 192 - Training Started...

Epoch 192, Batch 100, Loss: 0.0824, Accuracy: 95.53%, Time Passed: 4.31m
Epoch 192, Batch 200, Loss: 0.0859, Accuracy: 95.86%, Time Passed: 8.56m
Epoch 192, Batch 300, Loss: 0.0876, Accuracy: 95.91%,

In [71]:
if training_states:
    joblib.dump(training_states, f'Training outputs/training_states_lowest_30_{start_epoch}_{end_epoch}.pkl')

In [72]:
del training_states

In [73]:
start_epoch = 201
end_epoch = 220
start_epoch, end_epoch

(201, 220)

In [74]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_lowest_30, full_web_lowest_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_lowest_30', save_dir = 'Training outputs',
                         class_weights='adjusted_norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 201 - Training Started...

Epoch 201, Batch 100, Loss: 0.0700, Accuracy: 96.53%, Time Passed: 4.02m
Epoch 201, Batch 200, Loss: 0.0704, Accuracy: 96.62%, Time Passed: 7.84m
Epoch 201, Batch 300, Loss: 0.0720, Accuracy: 96.71%, Time Passed: 11.67m
Epoch 201, Batch 400, Loss: 0.0754, Accuracy: 96.47%, Time Passed: 15.63m
Epoch 201, Batch 500, Loss: 0.0740, Accuracy: 96.46%, Time Passed: 19.68m
Epoch 201 Completed - Average Loss: 0.0748, Accuracy: 96.48%, Epoch Time: 22.56m
Training Complete Epoch 201 - Total Time: 22.56m

✅ Epoch 201 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.4801, Accuracy: 86.91%, Time Passed: 4.23m
Evaluation Complete - Loss: 0.4437, Accuracy: 88.28%, Total Time: 7.74m

📊 Epoch 201 - Evaluation Completed.


🚀 Epoch 202 - Training Started...

Epoch 202, Batch 100, Loss: 0.0761, Accuracy: 96.16%, Time Passed: 4.39m
Epoch 202, Batch 200, Loss: 0.0711, Accuracy: 96.64%, Time Passed: 9.09m
Epoch 202, Batch 300, Loss: 0.0738, Accuracy: 96.48%,

In [75]:
if training_states:
    joblib.dump(training_states, f'Training outputs/training_states_lowest_30_{start_epoch}_{end_epoch}.pkl')

In [76]:
del training_states

### final step

In [104]:
state, generator = setup_reproducibility(start_fresh=True, seed=seeds[0])

✅ Random seed set to 42 (MPS available: True)


In [105]:
training_states = joblib.load("Training outputs/training_states_lowest_30_201_220.pkl")
mobilenetv2_stageF_lowest_30 = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes)) 
device = torch.device("mps") 
mobilenetv2_stageF_lowest_30.to(device)
mobilenetv2_stageF_lowest_30.load_state_dict(training_states[215]["model_state_dict"])
del training_states

In [106]:
default_train_transform = build_train_transform(crop_type="random", scale=(0.2, 1.0))
train_dataset.transform = default_train_transform  
web_lowest_30_dataset.transform = default_train_transform
val_dataset.transform = val_transform  
full_web_lowest_30_dataset = ConcatDataset([train_dataset, web_lowest_30_dataset])
train_dataset.transform

Compose(
      ToImage()
      RandomResizedCrop(size=(224, 224), scale=(0.2, 1.0), ratio=(0.75, 1.33), interpolation=InterpolationMode.BILINEAR, antialias=True)
      RandomHorizontalFlip(p=0.5)
      RandomVerticalFlip(p=0.2)
      RandomApply(    RandomRotation(degrees=[-90.0, 90.0], interpolation=InterpolationMode.BILINEAR, expand=False, fill=0))
      ToDtype(scale=True)
      RandomApply(    ColorJitter(brightness=(0.7, 1.3), contrast=(0.85, 1.3), saturation=(0.7, 1.3), hue=(-0.0278, 0.0278)))
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False)
)

In [107]:
full_web_lowest_30_loader = DataLoader(
    full_web_lowest_30_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,             # shuffle ensures different order each epoch
    num_workers=4,            # safer on macOS, avoids multiprocessing issues
    pin_memory=True,      
    generator=generator       # ensures reproducible shuffling
)

In [108]:
norm_sqrt_inv_weights_train = compute_class_weights(train_dataset, method="sqrt_inverse", normalize=True)
norm_sqrt_inv_weights_train_lowest_30 = compute_class_weights(web_lowest_30_dataset, method="sqrt_inverse", normalize=True)   
norm_sqrt_inv_weights_train_lowest_30[-1] = 0 # handel other class
norm_sqrt_inv_weights = (norm_sqrt_inv_weights_train + (2*norm_sqrt_inv_weights_train_lowest_30))/2


lowest_30_val_results201_220 = pd.read_csv("Training outputs/mobilenetv2_stageF_lowest_30_val_results201_220.csv")

# Step 1: get the subset dataframe
df = lowest_30_val_results201_220[
    (lowest_30_val_results201_220['Species'] != "Average Performance (Macro)") &
    (lowest_30_val_results201_220['Epoch'] == 215)
].copy()

# Step 2: build a mapping from species to recall multiplier
def get_multiplier(recall):
    if recall < 0.6:
        return 10
    elif recall < 0.7:
        return 8
    elif recall < 0.8:
        return 6
    elif recall < 0.9:
        return 4
    elif recall < 1:
        return 2
    else:
        return 1

df['Multiplier'] = df['Recall'].apply(get_multiplier)

# Step 3: make sure class order matches your dataset order
species_to_multiplier = dict(zip(df['Species'], df['Multiplier']))

# Step 4: apply the multipliers to your weight vector
adjusted_weights = norm_sqrt_inv_weights.clone() if hasattr(norm_sqrt_inv_weights, 'clone') else np.copy(norm_sqrt_inv_weights)

for idx, class_name in enumerate(train_dataset.classes):
    if class_name in species_to_multiplier:
        adjusted_weights[idx] *= species_to_multiplier[class_name]

In [109]:
df

,Species,Precision,Recall,F1-score,Epoch,Multiplier
476,Anabasis articulata (Forssk.) Moq.,0.885714,1.000000,0.939394,215,1
477,Anabasis setifera Moq.,0.902703,0.843434,0.872063,215,4
478,Atriplex halimus L.,0.773292,0.886121,0.825871,215,4
479,Calotropis procera,0.950000,0.938272,0.944099,215,2
480,Capparis spinosa L.,0.926829,0.802817,0.860377,215,4
481,Cebatha pendula (J.R.Forst. & G.Forst.) Kuntze,0.935185,0.841667,0.885965,215,4
482,Cenchrus divisus,0.976471,0.954023,0.965116,215,2
483,Deverra tortuosa (Desf.) DC.,0.909091,1.000000,0.952381,215,1
484,Deverra triradiata Hochst. ex Boiss.,0.859649,0.980000,0.915888,215,2
485,Diplotaxis harra (Forssk.) Boiss.,0.887218,0.867647,0.877323,215,4


In [110]:
adjusted_weights, norm_sqrt_inv_weights

(tensor([ 1.0964,  4.5686,  6.0324,  2.4340,  4.6086,  5.1000,  3.1388,  1.9039,
          4.6970,  4.2096,  3.3128,  8.3636,  6.3663,  1.9881,  2.0413,  2.8982,
          5.4200,  9.3887,  6.7435,  3.7124,  1.9145,  1.5909,  1.1301,  0.8475,
          2.5624,  2.1405, 11.5773,  6.6286,  0.8819,  2.4888,  3.3469,  2.6062,
          1.9130], device='mps:0'),
 tensor([1.0964, 1.1421, 1.5081, 1.2170, 1.1522, 1.2750, 1.5694, 1.9039, 2.3485,
         1.0524, 1.6564, 2.0909, 1.0611, 0.9941, 1.0207, 1.4491, 2.7100, 1.5648,
         1.1239, 0.9281, 0.9572, 1.5909, 1.1301, 0.8475, 1.2812, 1.0702, 1.4472,
         1.1048, 0.8819, 1.2444, 1.6734, 2.6062, 0.4783], device='mps:0'))

In [111]:
norm_sqrt_inv_criterion = nn.CrossEntropyLoss(weight=adjusted_weights)
stageF_optimizer = torch.optim.AdamW(mobilenetv2_stageF_lowest_30.parameters(), lr=5e-6,
                            weight_decay=1e-4, betas=(0.9, 0.999))

In [74]:
start_epoch = 216
end_epoch = 220
start_epoch, end_epoch

(216, 220)

In [113]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageF_lowest_30, full_web_lowest_30_loader, val_loader, norm_sqrt_inv_criterion, stageF_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageF_lowest_30', save_dir = 'Training outputs',
                         class_weights='adjusted_norm_sqrt_inv', log_interval=100, evaluate=True)


🚀 Epoch 216 - Training Started...

Epoch 216, Batch 100, Loss: 0.0558, Accuracy: 96.91%, Time Passed: 3.70m
Epoch 216, Batch 200, Loss: 0.0636, Accuracy: 96.94%, Time Passed: 7.37m
Epoch 216, Batch 300, Loss: 0.0651, Accuracy: 96.95%, Time Passed: 11.76m
Epoch 216, Batch 400, Loss: 0.0660, Accuracy: 96.90%, Time Passed: 16.07m
Epoch 216, Batch 500, Loss: 0.0641, Accuracy: 97.01%, Time Passed: 20.41m
Epoch 216 Completed - Average Loss: 0.0629, Accuracy: 97.05%, Epoch Time: 23.45m
Training Complete Epoch 216 - Total Time: 23.45m

✅ Epoch 216 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.4612, Accuracy: 87.44%, Time Passed: 4.44m
Evaluation Complete - Loss: 0.4328, Accuracy: 88.84%, Total Time: 7.88m

📊 Epoch 216 - Evaluation Completed.


🚀 Epoch 217 - Training Started...

Epoch 217, Batch 100, Loss: 0.0550, Accuracy: 97.28%, Time Passed: 4.53m
Epoch 217, Batch 200, Loss: 0.0555, Accuracy: 97.28%, Time Passed: 8.75m
Epoch 217, Batch 300, Loss: 0.0601, Accuracy: 97.18%,

In [75]:
if training_states:
    joblib.dump(
        training_states,
        f"Training outputs/training_states_lowest_30_{start_epoch}_{end_epoch}.pkl"
    )